# Dynamic active-site identification on Fe(111)

**Student-friendly complete version.**

This notebook teaches how to identify dynamic active sites on Fe(111) by combining:

```text
ASE → OVITO → PLUMED → OPES reweighting → committor analysis → SOAP clustering
```

The notebook is organized in three levels.

| Level | Sections | Purpose |
|---|---|---|
| **Core workflow** | 1–13 | Identify surface atoms, compute \(S(\chi,\chi_7)\), count and characterize \(\chi_7\)-like motifs. |
| **Reactive trajectories** | 14 | Analyze N\(_2\) activation, OPES reweighting, and committor-selected TS snapshots. |
| **Unsupervised discovery** | 15 | Use SOAP + PCA + clustering to discover local environments without imposing \(\chi_7\). |

For a classroom run, keep:

```python
MAX_REACTIVE_FRAMES = 250
```

For the full analysis, set it to:

```python
MAX_REACTIVE_FRAMES = None
```

The reactive trajectories were generated with OPES bias. Therefore, reactive **populations and distributions** are reweighted, while reactive **lifetimes** are not interpreted as physical quantities.

## 0. Scientific idea

The active site is not defined as one fixed atom or one static adsorption site. Instead, we define a reference local environment $\chi_7$ and ask, for every Fe atom and every MD frame:

$$
S(\chi,\chi_7) =
\frac{1}{n}
\sum_{i\in\chi}
\sum_{j\in\chi_7}
\exp\left[
-\frac{|\mathbf{r}_i-\mathbf{r}_j^{0}|^2}{4\sigma^2}
\right]
$$

If $ (S(\chi,\chi_7) \geq 0.8) $, the Fe atom is classified as $\chi_7$-like.

The comparison is between:

- **300 K:** ordered surface, stable $\chi_7$ motifs;
- **700 K:** rough and dynamic surface, transient $\chi_7$ motifs.


## 1. Imports, paths, and mandatory PLUMED check

Use the general `compcatschool` environment described in the repository-level `README.md`. This tutorial does not require a separate environment.

Start Jupyter from the `4_active_sites` directory so that the relative paths below resolve correctly. PLUMED must be available in `PATH` and include the `envsim` and `opes` modules.

The tutorial assumes this structure:

```text
4_active_sites/
  active_sites.ipynb
  data/
    alphaprime_environment_bulk.xyz
    300K/
      traj_ovito_300K_stride100.xyz
    700K/
      traj_ovito_700K_stride100.xyz
    N2/
      300K/traj-charges_N2_300K_stride100.extxyz
      700K/traj-charges_N2_700K_stride100.extxyz
      300K/opes_weights_300K.dat
      700K/opes_weights_700K.dat
    committor/
      snapshots_300K.tar.gz
      snapshots_700K.tar.gz
```

The notebook recomputes the two key descriptors used in the tutorial:

```text
surface atoms        → OVITO
χ7 similarity        → PLUMED
```

Any pre-existing `EnvSimilarity` or `SelectionSurface` arrays are treated only as diagnostics or cache data.

The notebook generates surface-selected trajectories, PLUMED working files, extracted committor snapshots, and SOAP exports beside these inputs. These files are caches and are excluded from version control. Use the relevant `FORCE_RECOMPUTE_*` option to regenerate them.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import tarfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ase.io import read, iread, write

# The notebook is meant to be run from the 4_active_sites folder.
# If you opened it elsewhere, move to the folder containing this notebook first.
ROOT = Path.cwd()
DATA = ROOT / "data"

REF_FILE = DATA / "alphaprime_environment_bulk.xyz"

SAMPLE_300K = DATA / "300K" / "traj_ovito_300K_stride100.xyz"
SAMPLE_700K = DATA / "700K" / "traj_ovito_700K_stride100.xyz"


PLUMED_DIR = ROOT / "plumed"
PLUMED_DIR.mkdir(exist_ok=True)

THRESHOLD = 0.80

# Sigma values used in Bonati et al. and in the supporting-analysis script.
SIGMA_BY_T = {
    300: 0.15,
    400: 0.15,
    500: 0.15,
    600: 0.17,
    700: 0.185,
    800: 0.20,
}

print("Working directory:", ROOT)
print("Data directory exists:", DATA.exists())
print("Reference file exists:", REF_FILE.exists())
print("300 K sample exists:", SAMPLE_300K.exists())
print("700 K sample exists:", SAMPLE_700K.exists())

if shutil.which("plumed") is None:
    raise RuntimeError(
        "PLUMED is required for this tutorial but was not found in PATH. "
        "Install or activate PLUMED, then restart the notebook."
    )

plumed_executable = shutil.which("plumed")
print("PLUMED executable:", plumed_executable)

version_check = subprocess.run(
    ["plumed", "info", "--version"],
    capture_output=True,
    text=True,
)
if version_check.returncode != 0:
    raise RuntimeError(version_check.stderr.strip() or "Could not determine the PLUMED version.")
print("PLUMED version:", version_check.stdout.strip())

required_plumed_actions = ["ENVIRONMENTSIMILARITY", "OPES_METAD"]
missing_plumed_actions = []
for action in required_plumed_actions:
    result = subprocess.run(
        ["plumed", "manual", "--action", action],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        missing_plumed_actions.append(action)

if missing_plumed_actions:
    raise RuntimeError(
        "The PLUMED executable selected from PATH does not provide: "
        + ", ".join(missing_plumed_actions)
        + ". Source a PLUMED build configured with --enable-modules=all "
        "before starting Jupyter, then restart the kernel. "
        f"Selected executable: {plumed_executable}"
          """If the wrong PLUMED is selected, adjust your PATH to prioritize the correct one.
import os
plumed_root = PATH_TO_PLUMED"/plumed-2.10.0"
os.environ["PATH"] = f"{plumed_root}/src/lib:" + os.environ["PATH"]
os.environ["PLUMED_KERNEL"] = f"{plumed_root}/src/lib/libplumedKernel.dylib"
"""
    )

print("Required PLUMED actions: OK")

## Sigma convention used in this tutorial

For the actual active-site analysis, we use the broadening parameters reported in Bonati *et al.*:

```text
300 K, 400 K, 500 K : sigma = 0.15 Å
600 K               : sigma = 0.17 Å
700 K               : sigma = 0.185 Å
800 K               : sigma = 0.20 Å
```

The stored `EnvSimilarity` arrays in the provided trajectories are used only as a diagnostic reference. If a stored trajectory was post-processed with a different `sigma`, the PLUMED/stored validation plot will not fall perfectly on the diagonal. That does not affect the tutorial workflow: PLUMED recomputes the descriptor with the paper values above, and the analysis uses those PLUMED-computed values.


## 2. Load the $\chi_7$ reference environment with ASE

ASE is used to load `alphaprime_environment_bulk.xyz` and extract the reference neighbor vectors.

ASE uses zero-based indexing. The original supporting-information script used `central_atom = 4`, i.e. the fifth atom in the file.


In [ ]:
ref_atoms = read(REF_FILE)
central_atom = 4
neighbor_indices = [i for i in range(len(ref_atoms)) if i != central_atom]

ref_vecs = ref_atoms.get_distances(
    central_atom,
    neighbor_indices,
    mic=True,
    vector=True,
)

print(ref_atoms)
print("Number of reference neighbor vectors:", len(ref_vecs))
print(ref_vecs[:3])



In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 4.2), dpi=140)

ax.scatter([0], [0], s=180, marker="*", label="central Fe")
ax.scatter(ref_vecs[:, 0], ref_vecs[:, 1], s=70, label=r"$\chi_7$ neighbors")

for x, y, z in ref_vecs:
    ax.plot([0, x], [0, y], lw=0.6)

ax.set_aspect("equal")
ax.set_xlabel(r"$x$ / Å")
ax.set_ylabel(r"$y$ / Å")
ax.set_title(r"2D projection of the reference $\chi_7$ environment")
ax.legend(frameon=False)
plt.show()


## 3. Write the PLUMED reference and input files

PLUMED's `ENVIRONMENTSIMILARITY` action with `CRYSTAL_STRUCTURE=CUSTOM` expects a PDB file containing the distance vectors from the central atom to the reference neighbors.

For each temperature we write a PLUMED input file that computes:

- the environment-similarity multicolvar `es`;
- `es.mean`;
- `es.morethan`, a smooth count of environments above the threshold;
- a `DUMPMULTICOLVAR` file containing the per-atom similarity values.


In [ ]:
def write_plumed_reference_pdb(ref_vecs, outfile, atom_name="FE", residue="CHI"):
    """Write the reference neighbor vectors in the PDB-like format expected by PLUMED."""
    outfile = Path(outfile)
    with outfile.open("w") as f:
        for i, (x, y, z) in enumerate(ref_vecs, start=1):
            f.write(
                f"ATOM  {i:5d} {atom_name:<4s} {residue:>3s} A   1"
                f"    {x:8.3f}{y:8.3f}{z:8.3f}"
                f"  1.00  0.00          FE\n"
            )
        f.write("END\n")
    return outfile


def write_plumed_envsim_input(outfile, reference_pdb, n_atoms=768, sigma=0.15, threshold=0.80):
    """Write the PLUMED input used to compute ENVIRONMENTSIMILARITY."""
    outfile = Path(outfile)
    lines = [
        "UNITS LENGTH=A TIME=fs",
        "FLUSH STRIDE=1",
        (
            "es: ENVIRONMENTSIMILARITY "
            f"SPECIES=1-{n_atoms} "
            f"SIGMA={sigma} "
            "CRYSTAL_STRUCTURE=CUSTOM "
            f"REFERENCE={reference_pdb.name} "
            f"MEAN MORE_THAN={{RATIONAL R_0={threshold} NN=12 MM=24}}"
        ),
        "PRINT STRIDE=1 ARG=es.mean,es.morethan FILE=COLVAR",
        "DUMPMULTICOLVAR DATA=es FILE=envsim_multicolvar.xyz",
    ]
    outfile.write_text("\n".join(lines) + "\n")
    return outfile


plumed_inputs = {}

for T in [300, 700]:
    workdir = PLUMED_DIR / f"{T}K"
    workdir.mkdir(parents=True, exist_ok=True)

    ref_pdb = write_plumed_reference_pdb(ref_vecs, workdir / "chi7_reference.pdb")
    plumed_dat = write_plumed_envsim_input(
        workdir / "plumed.dat",
        ref_pdb,
        n_atoms=768,
        sigma=SIGMA_BY_T[T],
        threshold=THRESHOLD,
    )
    plumed_inputs[T] = (workdir, ref_pdb, plumed_dat)

work300, ref300, plumed300 = plumed_inputs[300]
work700, ref700, plumed700 = plumed_inputs[700]

print("300 K PLUMED input:")
print(plumed300.read_text())
print("700 K PLUMED input:")
print(plumed700.read_text())



## 4. Load the trajectories with ASE

ASE is used to read the smoothed extended-XYZ trajectories.

The input files may already contain arrays such as `SelectionSurface` and `EnvSimilarity`, produced by previous post-processing. In this notebook we intentionally **ignore** them for the non-reactive trajectories:

- the surface atoms are selected again with OVITO;
- the environment similarity is recomputed with PLUMED.

The helper function `write_xyz_for_plumed_with_box(...)` is also defined here. It writes an XYZ trajectory in the format expected by `plumed driver`, with the simulation box on the second line of each frame.

In [ ]:
def load_ase_trajectory(path):
    """Load all frames from an ASE-readable trajectory."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return list(iread(path, index=":"))


def write_xyz_for_plumed_with_box(ase_traj, outfile):
    """Write an XYZ trajectory using PLUMED's simple box-line convention."""
    outfile = Path(outfile)
    outfile.parent.mkdir(parents=True, exist_ok=True)

    with outfile.open("w") as f:
        for atoms in ase_traj:
            if atoms.cell is None or atoms.cell.rank < 3:
                raise ValueError("Each frame must contain a 3D simulation cell for PLUMED.")

            cell = np.asarray(atoms.cell.array, dtype=float)
            lengths = atoms.cell.lengths()
            if np.allclose(cell, np.diag(lengths), atol=1e-8):
                box_line = " ".join(f"{x:.10f}" for x in lengths)
            else:
                box_line = " ".join(f"{x:.10f}" for x in cell.reshape(-1))

            f.write(f"{len(atoms)}\n")
            f.write(box_line + "\n")

            for symbol, (x, y, z) in zip(atoms.get_chemical_symbols(), atoms.get_positions()):
                f.write(f"{symbol:2s} {x:16.10f} {y:16.10f} {z:16.10f}\n")

    return outfile


# These input trajectories are already smoothed, but in this version we do not
# trust or use any pre-existing SelectionSurface or EnvSimilarity arrays.
traj300_input = load_ase_trajectory(SAMPLE_300K)
traj700_input = load_ase_trajectory(SAMPLE_700K)

print("300 K input frames:", len(traj300_input), SAMPLE_300K)
print("700 K input frames:", len(traj700_input), SAMPLE_700K)
print("Arrays present in the 300 K input file:", list(traj300_input[0].arrays.keys()))



## 5. Recompute the surface-atom selection with the OVITO Python library

In the paper, the environment-similarity analysis is restricted to dynamically identified **surface atoms**. Here we recompute that selection with OVITO so the criterion is visible in the notebook.

The workflow is deliberately simple:

```text
trajectory
  -> keep Fe atoms
  -> OVITO Alpha-Shape surface detection
  -> OVITO ExpressionSelection: Selection && Position.Z > z_cutoff
  -> store the result as SelectionSurface
```

`Selection` is the surface mask produced by OVITO. The expression `Position.Z > z_cutoff` keeps only the upper half of the slab. By default, `z_cutoff` is the mid-plane of the first processed frame; if your trajectory is centered differently, set `UPPER_SURFACE_Z_CUTOFF` manually.



In [ ]:
UPPER_SURFACE_Z_CUTOFF = None
# Example if your slab is centered around z = 0:
# UPPER_SURFACE_Z_CUTOFF = 0.0


def load_surface_trajectory(path):
    """Load a trajectory and require a SelectionSurface array in every frame."""
    traj = load_ase_trajectory(path)
    missing = [i for i, atoms in enumerate(traj) if "SelectionSurface" not in atoms.arrays]
    if missing:
        raise KeyError(f"SelectionSurface is missing from frames {missing[:10]}.")
    return traj


def add_upper_surface_selection_with_ovito(
    input_file,
    output_file,
    alpha_radius=2.0,
    z_cutoff=UPPER_SURFACE_Z_CUTOFF,
    start_frame=0,
    stop_frame=None,
    stride=1,
):
    """Use OVITO to select exposed Fe atoms on the upper surface of the slab."""
    try:
        from ovito.io import import_file
        from ovito.modifiers import (
            ConstructSurfaceModifier,
            ExpressionSelectionModifier,
            SelectTypeModifier,
        )
        from ovito.io.ase import ovito_to_ase
    except ImportError as exc:
        raise ImportError(
            "This function requires the OVITO Python module. "
            "Install the active-sites dependencies listed in the repository README."
        ) from exc

    input_file = Path(input_file)
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    pipeline = import_file(str(input_file), sort_particles=True)
    pipeline.modifiers.append(SelectTypeModifier(property="Particle Type", types={"Fe"}))
    pipeline.modifiers.append(
        ConstructSurfaceModifier(
            only_selected=True,
            method=ConstructSurfaceModifier.Method.AlphaShape,
            radius=alpha_radius,
            select_surface_particles=True,
            smoothing_level=1,
        )
    )

    if stop_frame is None:
        stop_frame = pipeline.source.num_frames

    if z_cutoff is None:
        first_data = pipeline.compute(start_frame)
        z = np.asarray(first_data.particles["Position"].array)[:, 2]
        z_cutoff = 0.5 * (z.min() + z.max())

    expression = f"Selection && Position.Z > {float(z_cutoff):.8f}"
    pipeline.modifiers.append(ExpressionSelectionModifier(expression=expression))

    frames = []
    for frame in range(start_frame, stop_frame, stride):
        data = pipeline.compute(frame)
        atoms = ovito_to_ase(data)
        atoms.arrays["SelectionSurface"] = np.asarray(
            data.particles["Selection"].array,
            dtype=np.int32,
        )

        keep = {"numbers", "positions", "SelectionSurface", "net_charges"}
        for name in list(atoms.arrays):
            if name not in keep:
                del atoms.arrays[name]

        frames.append(atoms)

    write(output_file, frames, format="extxyz")
    print("OVITO upper-surface expression:", expression)
    return output_file


FORCE_RECOMPUTE_NONREACTIVE_SURFACE = False

surface_inputs = {
    300: SAMPLE_300K,
    700: SAMPLE_700K,
}
surface_outputs = {
    T: DATA / f"{T}K" / f"traj_ovito_{T}K_stride100_surface_recomputed.extxyz"
    for T in surface_inputs
}
surface_trajs = {}

for T, input_file in surface_inputs.items():
    output_file = surface_outputs[T]
    regenerate = FORCE_RECOMPUTE_NONREACTIVE_SURFACE or not output_file.exists()

    if not regenerate:
        try:
            surface_trajs[T] = load_surface_trajectory(output_file)
            print(f"{T} K: using cached surface trajectory:", output_file)
        except Exception as exc:
            print(f"{T} K: cached surface trajectory is not usable ({type(exc).__name__}); regenerating.")
            regenerate = True

    if regenerate:
        print(f"{T} K: computing upper-surface selection with OVITO...")
        add_upper_surface_selection_with_ovito(
            input_file=input_file,
            output_file=output_file,
            alpha_radius=2.0,
        )
        surface_trajs[T] = load_surface_trajectory(output_file)

traj300 = surface_trajs[300]
traj700 = surface_trajs[700]
surface_file_300 = surface_outputs[300]
surface_file_700 = surface_outputs[700]

xyz300 = write_xyz_for_plumed_with_box(traj300, work300 / "traj_for_plumed.xyz")
xyz700 = write_xyz_for_plumed_with_box(traj700, work700 / "traj_for_plumed.xyz")

print("300 K frames:", len(traj300), xyz300)
print("700 K frames:", len(traj700), xyz700)


## 6. Run PLUMED driver

This is the mandatory computation of the environment-similarity descriptor used in the rest of the tutorial.


In [ ]:
def run_plumed_driver(workdir, xyz_file, plumed_dat):
    workdir = Path(workdir)
    xyz_file = Path(xyz_file)
    plumed_dat = Path(plumed_dat)

    for fname in ["COLVAR", "envsim_multicolvar.xyz"]:
        f = workdir / fname
        if f.exists():
            f.unlink()

    cmd = [
        "plumed",
        "driver",
        "--plumed", str(plumed_dat.name),
        "--ixyz", str(xyz_file.name),
        "--length-units", "A",
    ]

    print("Running in", workdir)
    print(" ".join(cmd))
    result = subprocess.run(cmd, cwd=workdir, capture_output=True, text=True)

    if result.stdout:
        print(result.stdout[-2000:])
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"PLUMED driver failed in {workdir}")

    colvar = workdir / "COLVAR"
    multicolvar = workdir / "envsim_multicolvar.xyz"

    if not colvar.exists():
        raise FileNotFoundError(colvar)
    if not multicolvar.exists():
        raise FileNotFoundError(multicolvar)

    return colvar, multicolvar


colvar300, multicolvar300 = run_plumed_driver(work300, xyz300, plumed300)
colvar700, multicolvar700 = run_plumed_driver(work700, xyz700, plumed700)

print("PLUMED outputs:")
print(colvar300)
print(multicolvar300)
print(colvar700)
print(multicolvar700)


## 7. Read PLUMED output

`COLVAR` contains aggregate quantities (`es.mean`, `es.morethan`).
`envsim_multicolvar.xyz` contains one similarity value per Fe atom.

For this tutorial we use the simple ordering assumption: PLUMED writes the per-atom rows in the same order as the Fe atoms in the ASE trajectory. This follows from using `SPECIES=1-768` for trajectories where the Fe atoms are the first 768 atoms.



In [ ]:
def read_plumed_colvar(path, temp):
    """Read a PLUMED COLVAR file into a tidy table."""
    path = Path(path)
    header = None
    rows = []

    with path.open() as f:
        for line in f:
            if line.startswith("#! FIELDS"):
                header = line.split()[2:]
            elif line.strip() and not line.startswith("#"):
                rows.append([float(x) for x in line.split()])

    if header is None:
        raise ValueError(f"No '#! FIELDS' header found in {path}")

    df = pd.DataFrame(rows, columns=header)
    df["frame"] = np.arange(len(df))
    df["T"] = temp
    return df


def read_plumed_multicolvar_xyz(path, temp):
    """Read per-atom similarity values from PLUMED DUMPMULTICOLVAR output."""
    rows = []
    frame = 0

    with Path(path).open() as f:
        while True:
            first = f.readline()
            if not first:
                break
            if not first.strip():
                continue

            n_atoms = int(first)
            _comment = f.readline()

            for row_id in range(n_atoms):
                parts = f.readline().split()
                if not parts:
                    raise ValueError(f"Empty atom line in {path}, frame {frame}.")

                rows.append(
                    {
                        "T": temp,
                        "frame": frame,
                        "row_id": row_id,
                        "S_plumed": float(parts[-1]),
                    }
                )

            frame += 1

    return pd.DataFrame(rows)


def assign_plumed_rows_to_fe_atoms(plumed_table, ase_traj, temp):
    """Assign PLUMED rows to Fe atoms using the shared trajectory order."""
    assigned_frames = []

    for frame, atoms in enumerate(ase_traj):
        pframe = (
            plumed_table[plumed_table["frame"] == frame]
            .sort_values("row_id")
            .reset_index(drop=True)
            .copy()
        )

        symbols = np.array(atoms.get_chemical_symbols())
        fe_ids = np.where(symbols == "Fe")[0]

        if len(pframe) != len(fe_ids):
            raise ValueError(
                f"Frame {frame}, T={temp} K: PLUMED wrote {len(pframe)} rows, "
                f"but ASE found {len(fe_ids)} Fe atoms."
            )

        pframe["atom_id"] = fe_ids
        pframe["T"] = temp
        assigned_frames.append(pframe)

    return pd.concat(assigned_frames, ignore_index=True)


col300 = read_plumed_colvar(colvar300, 300)
col700 = read_plumed_colvar(colvar700, 700)
plumed_colvar = pd.concat([col300, col700], ignore_index=True)

raw300 = read_plumed_multicolvar_xyz(multicolvar300, 300)
raw700 = read_plumed_multicolvar_xyz(multicolvar700, 700)

plumed300_values = assign_plumed_rows_to_fe_atoms(raw300, traj300, 300)
plumed700_values = assign_plumed_rows_to_fe_atoms(raw700, traj700, 700)
plumed_values = pd.concat([plumed300_values, plumed700_values], ignore_index=True)

display(plumed_colvar.head())
display(plumed_values.head())



## 8. Combine PLUMED similarity with the recomputed OVITO surface mask

The analysis below uses only quantities generated inside this notebook:

- `SelectionSurface` recomputed with OVITO;
- `S_plumed` recomputed with PLUMED.

Any `EnvSimilarity` or `SelectionSurface` arrays already present in the original XYZ files are not used for the non-reactive active-site analysis.

In [ ]:
surface_rows = []

for T, traj in [(300, traj300), (700, traj700)]:
    for frame, atoms in enumerate(traj):
        symbols = np.array(atoms.get_chemical_symbols())
        surface = atoms.arrays["SelectionSurface"].astype(bool)
        positions = atoms.get_positions()

        for atom_id in np.where(symbols == "Fe")[0]:
            x, y, z = positions[atom_id]
            surface_rows.append(
                {
                    "T": T,
                    "frame": frame,
                    "atom_id": int(atom_id),
                    "SelectionSurface": bool(surface[atom_id]),
                    "x": float(x),
                    "y": float(y),
                    "z": float(z),
                }
            )

surface_metadata = pd.DataFrame(surface_rows)

plumed_analysis_values = plumed_values.merge(
    surface_metadata,
    on=["T", "frame", "atom_id"],
    how="inner",
)

# This is the dataframe used for all surface active-site analyses below.
plumed_surface_values = plumed_analysis_values[
    plumed_analysis_values["SelectionSurface"]
].copy()

display(plumed_analysis_values.head())

print("Number of PLUMED values on recomputed surface atoms:")
display(plumed_surface_values.groupby(["T", "frame"]).size().groupby("T").describe())

print("Fraction of Fe atoms selected as upper-surface atoms:")
display(
    plumed_analysis_values
    .groupby(["T", "frame"])["SelectionSurface"]
    .mean()
    .groupby("T")
    .describe()
)



## 9. Smooth similarity distributions from PLUMED

From now on we use the **PLUMED-computed** similarity values, restricted to the **OVITO-recomputed** surface atoms.

Instead of plotting histograms, we estimate smooth probability densities with a one-dimensional Gaussian kernel density estimator. This follows the visual style used in Bonati *et al.*, where similarity distributions are shown as smooth normalized densities.

The KDE helper below is implemented directly with NumPy, so the notebook does not require `scipy`.

In [ ]:
def gaussian_kde_1d(values, grid=None, bandwidth=None, n_grid=400, value_range=None):
    """Simple one-dimensional Gaussian KDE implemented with NumPy.

    Parameters
    ----------
    values : array-like
        Input samples.
    grid : array-like or None
        Points where the density is evaluated. If None, a grid is built.
    bandwidth : float or None
        Gaussian kernel width. If None, Silverman's rule of thumb is used.
    n_grid : int
        Number of grid points if grid is None.
    value_range : tuple or None
        Range used to construct the grid and normalize the KDE.

    Returns
    -------
    grid, density : np.ndarray
        The density is normalized so that the integral over the plotted
        grid is equal to 1.
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        raise ValueError("Cannot compute KDE of an empty array.")

    if value_range is None:
        vmin, vmax = np.min(values), np.max(values)
        if np.isclose(vmin, vmax):
            vmin -= 0.5
            vmax += 0.5
        padding = 0.05 * (vmax - vmin)
        value_range = (vmin - padding, vmax + padding)

    if grid is None:
        grid = np.linspace(value_range[0], value_range[1], n_grid)
    else:
        grid = np.asarray(grid, dtype=float)

    if len(values) == 1:
        if bandwidth is None:
            bandwidth = 0.03 * (value_range[1] - value_range[0])
    else:
        if bandwidth is None:
            std = np.std(values, ddof=1)
            q25, q75 = np.percentile(values, [25, 75])
            iqr_sigma = (q75 - q25) / 1.349 if q75 > q25 else std
            sigma = min(std, iqr_sigma) if std > 0 and iqr_sigma > 0 else max(std, iqr_sigma)
            if sigma <= 0 or not np.isfinite(sigma):
                sigma = 0.1 * (value_range[1] - value_range[0])
            bandwidth = 0.9 * sigma * len(values) ** (-1 / 5)

    if bandwidth <= 0 or not np.isfinite(bandwidth):
        bandwidth = 0.03 * (value_range[1] - value_range[0])

    z = (grid[:, None] - values[None, :]) / bandwidth
    density = np.exp(-0.5 * z**2).sum(axis=1)
    density /= len(values) * bandwidth * np.sqrt(2 * np.pi)

    integral = np.trapezoid(density, grid)
    if integral > 0:
        density /= integral

    return grid, density


def plot_similarity_kde_by_temperature(
    table,
    value_col,
    ax=None,
    title=None,
    label_suffix="",
    grid=np.linspace(0.0, 1.05, 400),
    bandwidth=None,
):
    """Plot smooth KDE curves of a similarity descriptor grouped by temperature."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6.2, 3.8), dpi=140)

    for T, group in table.groupby("T"):
        values = group[value_col].dropna().to_numpy()
        if len(values) == 0:
            continue

        x, y = gaussian_kde_1d(
            values,
            grid=grid,
            bandwidth=bandwidth,
            value_range=(grid.min(), grid.max()),
        )
        ax.plot(x, y, lw=1.8, label=f"{T} K{label_suffix}")

    ax.axvline(THRESHOLD, ls="--", lw=1.0, label=r"$S=0.8$")
    ax.set_xlabel(r"Environment similarity $S(\chi,\chi_7)$")
    ax.set_ylabel("Probability density")
    if title is not None:
        ax.set_title(title)
    ax.legend(frameon=False)
    return ax


fig, ax = plt.subplots(figsize=(6.2, 3.8), dpi=140)

plot_similarity_kde_by_temperature(
    plumed_surface_values,
    value_col="S_plumed",
    ax=ax,
    title=r"Surface Fe atoms only: smooth KDE",
)

plt.show()

## 10. Active-site count from PLUMED per-atom values

We apply the same hard threshold used in the paper to the PLUMED-computed per-atom similarities, **after filtering for the recomputed OVITO surface atoms**:

\[
N_{\chi_7}(t) =
\sum_{i\in\mathrm{surface}}
\mathbf{1}\left[S_i^\mathrm{PLUMED}(\chi,\chi_7)\geq0.8
ight]
\]

No precomputed `EnvSimilarity` values from the XYZ file are used in this analysis.

In [ ]:
plumed_surface_values = plumed_surface_values.copy()
plumed_surface_values["active_plumed"] = plumed_surface_values["S_plumed"] >= THRESHOLD

counts_plumed = (
    plumed_surface_values.groupby(["T", "frame"])
    .agg(
        N_surface=("atom_id", "count"),
        N_chi7_plumed=("active_plumed", "sum"),
    )
    .reset_index()
)

# Keep this alias for later summary cells.
counts_compare = counts_plumed.copy()

fig, ax = plt.subplots(figsize=(7.0, 3.8), dpi=140)

for T, group in counts_plumed.groupby("T"):
    ax.plot(group["frame"], group["N_chi7_plumed"], lw=1.3, label=f"{T} K")

ax.set_xlabel("Stored frame")
ax.set_ylabel(r"$N_{\chi_7}(t)$")
ax.set_title(r"Active-site count from PLUMED similarity on recomputed surface atoms")
ax.legend(frameon=False)
plt.show()

counts_plumed.groupby("T")[["N_surface", "N_chi7_plumed"]].agg(["mean", "std", "min", "max"])

## 10b. Lifetime of $\chi_7$-like motifs in non-reactive trajectories

Here we estimate how long a given Fe atom remains $\chi_7$-like in consecutive stored frames.

This is done only for the **non-reactive** trajectories. These lifetimes are therefore not affected by OPES bias.

In [ ]:
def estimate_active_site_lifetimes(
    active_table,
    frame_col="frame",
    atom_col="atom_id",
    active_col="active_plumed",
    temp_col="T",
    time_between_frames_ps=None,
):
    """Estimate lifetimes of active-site motifs from consecutive active frames.

    The lifetime is atom-based: a segment starts when a given atom becomes
    active and ends when the same atom is no longer active, no longer present
    in the surface table, or when a frame gap occurs.
    """
    segments = []

    required = {temp_col, frame_col, atom_col, active_col}
    missing = required - set(active_table.columns)
    if missing:
        raise KeyError(f"Missing columns for lifetime analysis: {missing}")

    for (temp, atom_id), group in active_table.sort_values(frame_col).groupby([temp_col, atom_col]):
        frames = group[frame_col].to_numpy()
        active = group[active_col].to_numpy(dtype=bool)

        run_start = None
        run_end = None
        previous_frame = None

        for frame, is_active in zip(frames, active):
            consecutive = previous_frame is not None and frame == previous_frame + 1

            if is_active:
                if run_start is None or not consecutive:
                    if run_start is not None:
                        segments.append(
                            {
                                "T": temp,
                                "atom_id": atom_id,
                                "start_frame": run_start,
                                "end_frame": run_end,
                                "lifetime_frames": run_end - run_start + 1,
                            }
                        )
                    run_start = frame
                run_end = frame
            else:
                if run_start is not None:
                    segments.append(
                        {
                            "T": temp,
                            "atom_id": atom_id,
                            "start_frame": run_start,
                            "end_frame": run_end,
                            "lifetime_frames": run_end - run_start + 1,
                        }
                    )
                    run_start = None
                    run_end = None

            previous_frame = frame

        if run_start is not None:
            segments.append(
                {
                    "T": temp,
                    "atom_id": atom_id,
                    "start_frame": run_start,
                    "end_frame": run_end,
                    "lifetime_frames": run_end - run_start + 1,
                }
            )

    lifetimes = pd.DataFrame(segments)

    if len(lifetimes) == 0:
        return lifetimes

    if time_between_frames_ps is not None:
        lifetimes["lifetime_ps"] = [
            row["lifetime_frames"] * time_between_frames_ps[row["T"]]
            for _, row in lifetimes.iterrows()
        ]

    return lifetimes


# Set this to the physical time spacing of the trajectory being analyzed.
# Leave as None if you want to report lifetimes in consecutive stored frames.
TIME_BETWEEN_FRAMES_PS = None
# Example:
# TIME_BETWEEN_FRAMES_PS = {300: 200.0, 700: 200.0}

lifetimes = estimate_active_site_lifetimes(
    plumed_surface_values,
    time_between_frames_ps=TIME_BETWEEN_FRAMES_PS,
)

display(lifetimes.head())

if len(lifetimes) == 0:
    print("No active-site lifetime segments were found. Check the threshold or the active_plumed column.")
else:
    print("Lifetime summary:")
    display(lifetimes.groupby("T")["lifetime_frames"].agg(["count", "mean", "median", "min", "max"]))

    if "lifetime_ps" in lifetimes.columns:
        display(lifetimes.groupby("T")["lifetime_ps"].agg(["count", "mean", "median", "min", "max"]))

    value_col = "lifetime_ps" if "lifetime_ps" in lifetimes.columns else "lifetime_frames"

    # -----------------------------------------------------------------
    # Linear-scale survival plot
    # -----------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(6.2, 3.8), dpi=140)

    for T, group in lifetimes.groupby("T"):
        values = np.sort(group[value_col].dropna().to_numpy())
        values = values[values > 0]

        if len(values) == 0:
            print(f"{T} K: no lifetime values to plot.")
            continue

        tau = np.unique(values)
        survival = np.array([(values >= t).mean() for t in tau])

        label = f"{T} K, n={len(values)}"

        if len(tau) == 1:
            # Degenerate distribution: show it explicitly as a marker.
            ax.scatter(tau, survival, s=70, label=label)
        else:
            ax.step(tau, survival, where="post", lw=1.8, label=label)
            ax.scatter(tau, survival, s=18, alpha=0.7)

    ax.set_xlabel(
        "Lifetime / ps" if "lifetime_ps" in lifetimes.columns
        else "Lifetime / consecutive stored frames"
    )
    ax.set_ylabel(r"$P(\mathrm{lifetime} \geq \tau)$")
    ax.set_title("Survival function of active-site lifetimes")
    ax.set_ylim(-0.05, 1.05)
    ax.legend(frameon=False)
    plt.show()

## 11. PLUMED aggregate output: `es.mean` and `es.morethan`

PLUMED also directly outputs `es.mean` and `es.morethan`.

`es.morethan` is a **smooth** count defined by the switching function in the PLUMED input, while the previous cell used a hard threshold on the per-atom PLUMED values. Both are useful:

- hard threshold: closer to the definition used in the paper;
- smooth count: differentiable and suitable for enhanced sampling or biased simulations.


In [ ]:
display(plumed_colvar.head())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), dpi=140)

for T, group in plumed_colvar.groupby("T"):
    es_mean_col, es_morethan_col = None, None
    for col in group.columns:
        if "es" in col and "mean" in col:
            es_mean_col = col
        elif "es" in col and "morethan" in col:
            es_morethan_col = col
    if es_mean_col is None:
        raise KeyError(f"Cannot find an es.mean column. Available columns: {list(group.columns)}")
    if es_morethan_col is None:
        raise KeyError(f"Cannot find an es.morethan column. Available columns: {list(group.columns)}")

    axes[0].plot(group["frame"], group[es_mean_col], lw=1.2, label=f"{T} K")
    axes[1].plot(group["frame"], group[es_morethan_col], lw=1.2, label=f"{T} K")

axes[0].set_xlabel("Stored frame")
axes[0].set_ylabel("es.mean")
axes[0].set_title("Mean environment similarity from PLUMED")
axes[0].legend(frameon=False)

axes[1].set_xlabel("Stored frame")
axes[1].set_ylabel("es.morethan")
axes[1].set_title("Smooth count of high-similarity environments")
axes[1].legend(frameon=False)

plt.tight_layout()
plt.show()


## 12. Surface roughness with ASE

Surface roughness is computed from the `SelectionSurface` mask recomputed with OVITO:

\[
S_q(t) = \sqrt{
rac{1}{N_s(t)}\sum_{i=1}^{N_s(t)}(z_i-ar{z})^2}
\]

In [ ]:
roughness_rows = []

for T, traj in [(300, traj300), (700, traj700)]:
    for frame, atoms in enumerate(traj):
        surface = atoms.arrays["SelectionSurface"].astype(bool)
        z = atoms.positions[:, 2]
        roughness_rows.append(
            {
                "T": T,
                "frame": frame,
                "roughness_A": float(np.std(z[surface])),
                "N_surface": int(surface.sum()),
            }
        )

roughness = pd.DataFrame(roughness_rows)

fig, ax = plt.subplots(figsize=(7.0, 3.6), dpi=140)

for T, group in roughness.groupby("T"):
    ax.plot(group["frame"], group["roughness_A"], lw=1.2, label=f"{T} K")

ax.set_xlabel("Stored frame")
ax.set_ylabel(r"Surface roughness $S_q$ / Å")
ax.set_title("Surface roughness from ASE-loaded trajectories")
ax.legend(frameon=False)
plt.show()

roughness.groupby("T")[["N_surface", "roughness_A"]].agg(["mean", "std", "min", "max"])


## 13. Summary: PLUMED-derived comparison

This table summarizes the non-reactive analysis using PLUMED-computed similarity values and the OVITO-recomputed surface mask.

In [ ]:
plumed_summary = counts_compare.groupby("T")["N_chi7_plumed"].agg(["mean", "std", "min", "max"])
rough_summary = roughness.groupby("T")["roughness_A"].agg(["mean", "std", "min", "max"])

display(plumed_summary)
display(rough_summary)

comparison = pd.DataFrame(
    {
        "300 K": [
            plumed_summary.loc[300, "mean"],
            rough_summary.loc[300, "mean"],
        ],
        "700 K": [
            plumed_summary.loc[700, "mean"],
            rough_summary.loc[700, "mean"],
        ],
    },
    index=[
        r"Mean $N_{\chi_7}$ from PLUMED",
        "Mean surface roughness / Å",
    ],
)

display(comparison)

## 14. Reactive $N_2$ trajectories: active-site analysis only

In this final part we apply the same active-site analysis to pre-run reactive trajectories containing N$_2$ at **300 K** and **700 K**.

We do **not** reconstruct a free-energy surface here. The goal is only to ask an active-site question:

> When N$_2$ is present, is the local Fe environment below/around the molecule $\chi_7$-like?

Both reactive trajectories have a stride of 100 frames and contain the per-atom charge information directly in the extended-XYZ arrays:

```text
bader_charges
net_charges
```

Therefore, the notebook reads $q(N_2)$ directly from the two N atoms in each frame.

The analysis is divided into smaller steps:

```text
15.1 runtime options and input files
15.2 load and inspect the reactive trajectories
15.3 identify surface atoms with OVITO
15.4 compute S(chi, chi7) with PLUMED
15.5 identify the local Fe environment below/around N2
15.6 compare 300 K and 700 K
15.7 plot active-site descriptors, focusing on TS-like frames
```

The last graphical analysis focuses on transition-state-like configurations using:

$$
1.6 \leq d_{\mathrm{NN}} \leq 1.8\ \mathrm{\AA}
$$

This avoids mixing TS-like configurations with already dissociated high-charge frames.

### 14.1 Runtime options and input files

The reactive trajectories are larger than the surface-only tutorial trajectories. To keep the notebook usable in a classroom, this section can analyze only the first part of each reactive trajectory by default.

Set:

```python
MAX_REACTIVE_FRAMES = None
```

to analyze the full stride-100 trajectories.

The generated OVITO and PLUMED files are cached. If you rerun the notebook, the expensive steps should be skipped unless you set:

```python
FORCE_RECOMPUTE_REACTIVE = True
```

In [ ]:
# -------------------------------------------------------------------------
# 14.1 Runtime options and input files
# -------------------------------------------------------------------------

REACTIVE_N2_TRAJECTORIES = {
    300: DATA / "N2" / "300K" / "traj-charges_N2_300K_stride100.extxyz",
    700: DATA / "N2" / "700K" / "traj-charges_N2_700K_stride100.extxyz",
}

# The files are already stride-100 with respect to the original reactive trajectory.
REACTIVE_SOURCE_STRIDE = 100

# Classroom-friendly default. Set to None to analyze the full stride-100 trajectories.
MAX_REACTIVE_FRAMES = 250

# Use every stored frame. Increase to 2, 5, ... for a faster preview.
REACTIVE_FRAME_STEP = 1

# If True, regenerate OVITO and PLUMED outputs even if cached files exist.
FORCE_RECOMPUTE_REACTIVE = False

# Active-site and reaction-region thresholds.
HIGH_CHARGE_CUTOFF = 1.35
TS_MIN_DNN = 1.6
TS_MAX_DNN = 1.8

max_part = "full" if MAX_REACTIVE_FRAMES is None else f"first{MAX_REACTIVE_FRAMES}"
REACTIVE_SUFFIX = f"{max_part}_step{REACTIVE_FRAME_STEP}"

print("Reactive analysis suffix:", REACTIVE_SUFFIX)
print("MAX_REACTIVE_FRAMES:", MAX_REACTIVE_FRAMES)
print("REACTIVE_FRAME_STEP:", REACTIVE_FRAME_STEP)
print("TS-like dNN window / Å:", TS_MIN_DNN, TS_MAX_DNN)

for T, path in REACTIVE_N2_TRAJECTORIES.items():
    print(f"{T} K:", path, "exists:", path.exists())



### 14.2 Load and inspect the reactive trajectories

This cell reads only the frames selected by `MAX_REACTIVE_FRAMES` and `REACTIVE_FRAME_STEP`.

We keep a mapping between notebook frame indices and original trajectory frame indices:

```text
notebook frame 0 -> original frame 0
notebook frame 1 -> original frame 100
...
```

if `REACTIVE_FRAME_STEP = 1`.

The original frame index is stored in `reactive_source_frames`.

In [ ]:
reactive_raw_trajs = {}
reactive_stored_frame_indices = {}
reactive_source_frames = {}

for T, path in REACTIVE_N2_TRAJECTORIES.items():
    print(f"\nLoading reactive trajectory at {T} K")

    traj = []
    stored_indices = []

    for stored_index, atoms in enumerate(iread(path, index=":")):
        if stored_index % REACTIVE_FRAME_STEP != 0:
            continue

        traj.append(atoms)
        stored_indices.append(stored_index)

        if MAX_REACTIVE_FRAMES is not None and len(traj) >= MAX_REACTIVE_FRAMES:
            break

    if len(traj) == 0:
        raise RuntimeError(f"No frames were loaded from {path}.")

    reactive_raw_trajs[T] = traj
    reactive_stored_frame_indices[T] = stored_indices
    reactive_source_frames[T] = [i * REACTIVE_SOURCE_STRIDE for i in stored_indices]

    first = traj[0]
    symbols = np.array(first.get_chemical_symbols())

    print("Loaded frames:", len(traj))
    print("First stored-frame index:", stored_indices[0])
    print("Last stored-frame index:", stored_indices[-1])
    print("First original-frame index:", reactive_source_frames[T][0])
    print("Last original-frame index:", reactive_source_frames[T][-1])
    print("Atoms in first frame:", len(first))
    print("Number of Fe atoms:", np.sum(symbols == "Fe"))
    print("Number of N atoms:", np.sum(symbols == "N"))
    print("Arrays in first frame:", list(first.arrays.keys()))

    if "net_charges" not in first.arrays:
        raise KeyError(f"The reactive trajectory at {T} K does not contain 'net_charges'.")



### 14.3 Identify surface atoms in the reactive trajectories with OVITO

The reactive trajectories contain N atoms, but active sites are defined on the Fe surface.

This cell uses the OVITO helper defined earlier:

```python
add_upper_surface_selection_with_ovito(...)
```

to create a `SelectionSurface` array for every atom.

The output is cached using a filename that includes the frame selection, for example:

```text
traj-charges_N2_300K_stride100_surface_first250_step1.extxyz
```

If the cached file exists but is incomplete or corrupted, the notebook deletes it and regenerates it.

In [ ]:
reactive_trajs = {}

for T, raw_traj in reactive_raw_trajs.items():
    input_traj = REACTIVE_N2_TRAJECTORIES[T]
    temp_dir = DATA / "N2" / f"{T}K"
    surface_traj = temp_dir / f"traj-charges_N2_{T}K_stride100_surface_{REACTIVE_SUFFIX}.extxyz"

    if "SelectionSurface" in raw_traj[0].arrays:
        print(f"{T} K: raw reactive trajectory already contains SelectionSurface.")
        traj = raw_traj
    else:
        regenerate = FORCE_RECOMPUTE_REACTIVE or not surface_traj.exists()

        if not regenerate:
            try:
                traj = load_surface_trajectory(surface_traj)
                print(f"{T} K: using cached surface trajectory:", surface_traj)
            except Exception as exc:
                print(f"{T} K: cached surface trajectory is not usable ({type(exc).__name__}); regenerating.")
                regenerate = True

        if regenerate:
            print(f"{T} K: generating upper-surface selection with OVITO...")
            stop_frame = None
            if MAX_REACTIVE_FRAMES is not None:
                stop_frame = MAX_REACTIVE_FRAMES * REACTIVE_FRAME_STEP

            add_upper_surface_selection_with_ovito(
                input_file=input_traj,
                output_file=surface_traj,
                alpha_radius=2.0,
                start_frame=0,
                stop_frame=stop_frame,
                stride=REACTIVE_FRAME_STEP,
            )
            traj = load_surface_trajectory(surface_traj)

    if len(traj) != len(reactive_stored_frame_indices[T]):
        raise RuntimeError(
            f"{T} K: processed trajectory has {len(traj)} frames, "
            f"but expected {len(reactive_stored_frame_indices[T])}."
        )

    reactive_trajs[T] = traj

    print(f"{T} K: frames after surface selection:", len(traj))
    print(f"{T} K: arrays in first processed frame:", list(traj[0].arrays.keys()))



### 14.4 Compute environment similarity for the reactive trajectories with PLUMED

For each temperature, PLUMED computes $S(\chi,\chi_7)$ for the Fe atoms.

The PLUMED outputs are written to separate folders:

```text
plumed/N2_300K_first250_step1/
plumed/N2_700K_first250_step1/
```

This avoids overwriting the surface-only PLUMED outputs.

The outputs are cached. Rerunning the notebook should skip the expensive PLUMED-driver step unless:

```python
FORCE_RECOMPUTE_REACTIVE = True
```

In [ ]:
reactive_plumed_values = {}
reactive_plumed_colvars = {}

for T, traj in reactive_trajs.items():
    print(f"\nPreparing PLUMED calculation for reactive {T} K trajectory...")

    work_n2 = PLUMED_DIR / f"N2_{T}K_{REACTIVE_SUFFIX}"
    work_n2.mkdir(parents=True, exist_ok=True)

    ref_n2 = write_plumed_reference_pdb(ref_vecs, work_n2 / "chi7_reference.pdb")
    plumed_n2 = write_plumed_envsim_input(
        work_n2 / "plumed.dat",
        ref_n2,
        n_atoms=768,
        sigma=SIGMA_BY_T[T],
        threshold=THRESHOLD,
    )

    xyz_n2 = work_n2 / f"traj_N2_{T}K_for_plumed.xyz"

    if FORCE_RECOMPUTE_REACTIVE or not xyz_n2.exists():
        print(f"{T} K: writing PLUMED-compatible XYZ trajectory...")
        write_xyz_for_plumed_with_box(traj, xyz_n2)
    else:
        print(f"{T} K: using cached PLUMED-compatible XYZ:", xyz_n2)

    colvar_n2 = work_n2 / "COLVAR"
    multicolvar_n2 = work_n2 / "envsim_multicolvar.xyz"

    if FORCE_RECOMPUTE_REACTIVE or not (colvar_n2.exists() and multicolvar_n2.exists()):
        print(f"{T} K: running PLUMED driver...")
        colvar_n2, multicolvar_n2 = run_plumed_driver(
            work_n2,
            xyz_n2,
            plumed_n2,
        )
    else:
        print(f"{T} K: using cached PLUMED outputs.")

    raw_s_n2 = read_plumed_multicolvar_xyz(multicolvar_n2, T)
    s_n2 = assign_plumed_rows_to_fe_atoms(raw_s_n2, traj, T)

    reactive_plumed_values[T] = s_n2
    reactive_plumed_colvars[T] = read_plumed_colvar(colvar_n2, T)

    print(f"{T} K: PLUMED values:")
    display(s_n2.head())
    display(s_n2["S_plumed"].describe())


### 14.5 Identify the local Fe environment below/around $N_2$

For each frame we construct a compact descriptor table.

For each frame, the code:

1. identifies the two N atoms;
2. computes $d_{\mathrm{NN}}$;
3. computes $q(N_2)$ from the `net_charges` array;
4. finds Fe atoms within 3 Å of either N atom;
5. selects the lowest-$z$ Fe atom among these local candidates;
6. reads the PLUMED-computed similarity of that Fe atom:
   $$
   S(N_2,\chi_7)
   $$
7. classifies the local environment as $\chi_7$-like if $S(N_2,\chi_7)\ge0.8$.

We also define two chemically useful subsets:

```text
high-charge frames:
    |q(N2)| >= 1.35 e

TS-like frames:
    1.6 Å <= dNN <= 1.8 Å
```

The TS-like subset is the most useful one for the final active-site comparison, because high charge alone can also include frames where N$_2$ is already dissociated.

In [ ]:
def reactive_n2_descriptors(
    traj,
    plumed_table,
    temp,
    stored_frame_indices,
    source_frames,
    cutoff_to_n2=3.0,
):
    """Build a frame-by-frame active-site table for a reactive N2 trajectory.

    The selected local Fe atom follows the idea of the supporting-information
    script: identify the Fe atoms within cutoff_to_n2 Å of either N atom and
    select the one with the lowest z coordinate.

    The N2 charge is read directly from the per-atom net_charges array stored
    in the extended-XYZ trajectory.
    """
    rows = []

    # Quick lookup for PLUMED-computed similarity.
    s_lookup = plumed_table.set_index(["frame", "atom_id"])["S_plumed"].to_dict()

    for frame, atoms in enumerate(traj):
        symbols = np.array(atoms.get_chemical_symbols())
        positions = atoms.get_positions()

        n_ids = np.where(symbols == "N")[0]
        fe_ids = np.where(symbols == "Fe")[0]

        if len(n_ids) != 2:
            raise ValueError(f"Expected two N atoms in frame {frame}, found {len(n_ids)}")

        # Molecular descriptor: N-N distance.
        d_NN = atoms.get_distance(n_ids[0], n_ids[1], mic=True)

        # Electronic descriptor: total net charge on the two N atoms.
        # We store both signed and absolute values because different charge
        # conventions can use opposite signs for electron transfer.
        if "net_charges" not in atoms.arrays:
            raise KeyError("Missing net_charges in reactive trajectory.")

        q_N2_signed = float(atoms.arrays["net_charges"][n_ids].sum())
        abs_q_N2 = abs(q_N2_signed)

        # Local Fe environment around N2.
        distances = atoms.get_all_distances(mic=True)

        # Candidate Fe atoms close to either of the two N atoms.
        near_n2 = (distances[n_ids[:, None], fe_ids] < cutoff_to_n2).any(axis=0)
        candidate_fe = fe_ids[near_n2]

        if len(candidate_fe) == 0:
            site_atom_id = np.nan
            s_n2_chi7 = np.nan
            site_is_surface = False
        else:
            # Select the lowest-z Fe atom in the local N2 environment.
            # This mirrors the original supporting-script idea.
            site_atom_id = int(candidate_fe[np.argmin(positions[candidate_fe, 2])])
            s_n2_chi7 = float(s_lookup.get((frame, site_atom_id), np.nan))
            site_is_surface = bool(atoms.arrays["SelectionSurface"][site_atom_id])

        rows.append(
            {
                "T": temp,
                "frame": frame,
                "stored_frame": stored_frame_indices[frame],
                "source_stride": REACTIVE_SOURCE_STRIDE,
                "source_frame": source_frames[frame],
                "d_NN_A": float(d_NN),
                "q_N2_signed": q_N2_signed,
                "abs_q_N2": abs_q_N2,
                "site_atom_id": site_atom_id,
                "site_is_surface": site_is_surface,
                "S_N2_chi7": s_n2_chi7,
                "active_site_below_N2": bool(s_n2_chi7 >= THRESHOLD) if np.isfinite(s_n2_chi7) else False,
            }
        )

    return pd.DataFrame(rows)


reactive_site_tables = []

for T, traj in reactive_trajs.items():
    table = reactive_n2_descriptors(
        traj,
        reactive_plumed_values[T],
        temp=T,
        stored_frame_indices=reactive_stored_frame_indices[T],
        source_frames=reactive_source_frames[T],
        cutoff_to_n2=3.0,
    )
    reactive_site_tables.append(table)

n2_sites = pd.concat(reactive_site_tables, ignore_index=True)

# Charge-regime labels based on the absolute charge transfer.
n2_sites["charge_regime"] = pd.cut(
    n2_sites["abs_q_N2"],
    bins=[-np.inf, 0.5, HIGH_CHARGE_CUTOFF, np.inf],
    labels=["low charge", "medium charge", "high charge"],
)

# Transition-state-like region based only on the N-N distance.
n2_sites["ts_like"] = (
    (n2_sites["d_NN_A"] >= TS_MIN_DNN) &
    (n2_sites["d_NN_A"] <= TS_MAX_DNN)
)

# Combined selection, useful for optional diagnostics.
n2_sites["high_charge_ts_like"] = (
    (n2_sites["abs_q_N2"] >= HIGH_CHARGE_CUTOFF) &
    n2_sites["ts_like"]
)

display(n2_sites.head())

display(
    n2_sites
    .groupby("T")[["d_NN_A", "q_N2_signed", "abs_q_N2", "S_N2_chi7"]]
    .describe()
)

print("Number of TS-like frames:")
display(n2_sites.groupby("T")["ts_like"].sum())

print("Number of high-charge TS-like frames:")
display(n2_sites.groupby("T")["high_charge_ts_like"].sum())

### 14.6 Compare low- and high-temperature reactive trajectories

This cell summarizes the active-site statistics for the two reactive trajectories.

The main quantities are:

```text
mean_S_N2_chi7       average similarity of the local Fe site below/around N2
fraction_chi7_like   fraction of frames with S(N2,chi7) >= 0.8
mean_abs_q_N2        average absolute charge transfer to/from N2
mean_d_NN_A          average N-N distance
```

We compute these values for:

1. all frames;
2. high-charge frames;
3. TS-like frames with $1.6 \le d_{\mathrm{NN}} \le 1.8\ \mathrm{\AA}$;
4. high-charge TS-like frames.

In [ ]:
def summarize_reactive_subset(table, label):
    """Summarize a subset of the reactive N2 active-site table."""
    if len(table) == 0:
        print(f"{label}: no frames in this subset.")
        return pd.DataFrame()

    summary = (
        table
        .groupby("T")
        .agg(
            n_frames=("frame", "count"),
            mean_S_N2_chi7=("S_N2_chi7", "mean"),
            median_S_N2_chi7=("S_N2_chi7", "median"),
            fraction_chi7_like=("active_site_below_N2", "mean"),
            mean_abs_q_N2=("abs_q_N2", "mean"),
            mean_signed_q_N2=("q_N2_signed", "mean"),
            mean_d_NN_A=("d_NN_A", "mean"),
        )
    )

    print(label)
    display(summary)
    return summary


summary_all = summarize_reactive_subset(
    n2_sites,
    "All frames:"
)

summary_high_charge = summarize_reactive_subset(
    n2_sites[n2_sites["abs_q_N2"] >= HIGH_CHARGE_CUTOFF],
    f"High-charge frames only: |q(N2)| >= {HIGH_CHARGE_CUTOFF:.2f} e"
)

summary_ts_like = summarize_reactive_subset(
    n2_sites[n2_sites["ts_like"]],
    f"TS-like frames only: {TS_MIN_DNN:.1f} Å <= dNN <= {TS_MAX_DNN:.1f} Å"
)

summary_high_charge_ts_like = summarize_reactive_subset(
    n2_sites[n2_sites["high_charge_ts_like"]],
    (
        f"High-charge TS-like frames: "
        f"|q(N2)| >= {HIGH_CHARGE_CUTOFF:.2f} e and "
        f"{TS_MIN_DNN:.1f} Å <= dNN <= {TS_MAX_DNN:.1f} Å"
    )
)

### 14.7 Plot the reactive active-site descriptors

The plots below stay focused on active-site recognition.

We show:

1. time series of \(S(N_2,\chi_7)\);
2. smooth KDE distributions of \(S(N_2,\chi_7)\) over all selected reactive frames;
3. smooth KDE distributions of \(S(N_2,\chi_7)\) in the TS-like window only;
4. final scatter plot restricted to TS-like frames.

The final scatter does **not** include already dissociated high-charge frames, because it only keeps configurations with:

$$
1.6 \le d_{\mathrm{NN}} \le 1.8\ \mathrm{\AA}.
$$

In [ ]:
# -------------------------------------------------------------------------
# Plot 1: Time series of local chi7 similarity below/around N2
# -------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.0, 3.8), dpi=140)

for T, group in n2_sites.groupby("T"):
    ax.plot(
        group["source_frame"],
        group["S_N2_chi7"],
        lw=1.0,
        label=f"{T} K",
    )

ax.axhline(THRESHOLD, ls="--", lw=1.0, label=r"$S=0.8$")
ax.set_xlabel("Original trajectory frame")
ax.set_ylabel(r"$S(N_2,\chi_7)$")
ax.set_title(r"Local $\chi_7$ similarity below/around N$_2$")
ax.legend(frameon=False)
plt.show()


# -------------------------------------------------------------------------
# Plot 2: Smooth KDE distribution over all selected reactive frames
# -------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6.2, 3.8), dpi=140)

plot_similarity_kde_by_temperature(
    n2_sites,
    value_col="S_N2_chi7",
    ax=ax,
    title=r"All selected reactive frames: smooth KDE",
)

ax.set_xlabel(r"$S(N_2,\chi_7)$")
plt.show()


# -------------------------------------------------------------------------
# Plot 3: Smooth KDE distribution restricted to TS-like frames
# -------------------------------------------------------------------------

ts_data = n2_sites[n2_sites["ts_like"]].copy()

fig, ax = plt.subplots(figsize=(6.2, 3.8), dpi=140)

plot_similarity_kde_by_temperature(
    ts_data,
    value_col="S_N2_chi7",
    ax=ax,
    title=(
        rf"TS-like frames only: "
        rf"{TS_MIN_DNN:.1f} $\leq d_{{NN}} \leq$ {TS_MAX_DNN:.1f} Å"
    ),
    label_suffix=" TS-like",
)

ax.set_xlabel(r"$S(N_2,\chi_7)$")
plt.show()


# -------------------------------------------------------------------------
# Plot 4: Final TS-like scatter plot
# -------------------------------------------------------------------------
# This is the most chemically focused plot:
# it removes already-dissociated frames by filtering the N-N distance window.

fig, ax = plt.subplots(figsize=(6.0, 4.2), dpi=140)

for T, group in ts_data.groupby("T"):
    if len(group) == 0:
        continue

    ax.scatter(
        group["abs_q_N2"],
        group["S_N2_chi7"],
        s=18,
        alpha=0.55,
        label=f"{T} K TS-like",
    )

ax.axhline(THRESHOLD, ls="--", lw=1.0, label=r"$S=0.8$")
ax.axvline(HIGH_CHARGE_CUTOFF, ls=":", lw=1.0, label=rf"$|q(N_2)|={HIGH_CHARGE_CUTOFF}$ e")

ax.set_xlabel(r"$|q(N_2)|$ / e")
ax.set_ylabel(r"$S(N_2,\chi_7)$")
ax.set_title(r"TS-like region: does N$_2$ sample $\chi_7$-like sites?")
ax.legend(frameon=False)
plt.show()


# -------------------------------------------------------------------------
# Optional table: TS-like frames only
# -------------------------------------------------------------------------

display(
    ts_data[
        [
            "T",
            "source_frame",
            "d_NN_A",
            "q_N2_signed",
            "abs_q_N2",
            "S_N2_chi7",
            "active_site_below_N2",
            "site_atom_id",
            "site_is_surface",
        ]
    ].head(20)
)

### 14.7b OPES reweighting of reactive distributions

The reactive N$_2$ trajectories were generated with OPES-enhanced sampling. Therefore, simple histograms or KDEs of reactive descriptors are **biased-sampling diagnostics**, not equilibrium distributions.

To reconstruct an equilibrium-like distribution we use the OPES bias saved in the OPES/COLVAR file.

For a configuration sampled with a bias \(V_\mathrm{bias}\), the unbiased statistical weight is proportional to:

\[
w_i \propto \exp\left[\beta\left(V_i - c_i\right)\right]
\]

where \(V_i\) is `opes.bias`, \(c_i\) is `opes.rct`, and \(\beta = 1/(RT)\). In this notebook we assume PLUMED energies are in kJ/mol, so \(RT = 0.008314462618\,T\) kJ/mol.

The uploaded 300 K OPES file has been placed at:

```text
data/N2/300K/opes_weights_300K.dat
```

If you provide the 700 K file, save it as:

```text
data/N2/700K/opes_weights_700K.dat
```

and rerun this section.

In [ ]:
# -------------------------------------------------------------------------
# OPES reweighting for reactive N2 trajectories
# -------------------------------------------------------------------------

KB_KJMOL_PER_K = 0.00831446261815324

OPES_WEIGHT_FILES = {
    300: DATA / "N2" / "300K" / "opes_weights_300K.dat",
    700: DATA / "N2" / "700K" / "opes_weights_700K.dat",
}

# Usually we remove only the OPES bias.
# Set True only if you explicitly want to also unbias the wall/restraint term
# stored in dwall.bias.
INCLUDE_DWALL_BIAS_IN_REWEIGHTING = False


def read_opes_colvar(path):
    """Read a PLUMED OPES/COLVAR file with a '#! FIELDS ...' header."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    with path.open() as f:
        first = f.readline().strip()

    if first.startswith("#! FIELDS"):
        columns = first.split()[2:]
    else:
        raise ValueError(
            f"Could not find a '#! FIELDS' header in {path}. "
            "Please check that this is a PLUMED COLVAR/OPES file."
        )

    table = pd.read_csv(
        path,
        sep=r"\s+",
        comment="#",
        names=columns,
        engine="python",
    )

    return table


def compute_opes_logweights(opes_table, temperature, include_dwall=False):
    """Compute relative OPES log-weights.

    The default is:
        log w = (opes.bias - opes.rct) / (R T)

    If include_dwall=True and 'dwall.bias' is present:
        log w = (opes.bias - opes.rct + dwall.bias) / (R T)

    The returned values are shifted by their maximum for numerical stability.
    """
    required = {"opes.bias", "opes.rct"}
    missing = required - set(opes_table.columns)

    if missing:
        raise KeyError(f"Missing required OPES columns: {missing}")

    rt = KB_KJMOL_PER_K * float(temperature)

    total_bias = opes_table["opes.bias"].to_numpy(dtype=float) - opes_table["opes.rct"].to_numpy(dtype=float)

    if include_dwall:
        if "dwall.bias" not in opes_table.columns:
            raise KeyError("include_dwall=True but 'dwall.bias' is not present.")
        total_bias = total_bias + opes_table["dwall.bias"].to_numpy(dtype=float)

    logw = total_bias / rt
    logw = logw - np.nanmax(logw)

    return logw


def attach_opes_weights_to_n2_sites(
    n2_site_table,
    opes_tables_by_temperature,
    include_dwall=False,
):
    """Attach OPES weights to the reactive descriptor table.

    Alignment strategy
    ------------------
    The reactive extxyz files are already stride-100 with respect to the original
    reactive trajectory. The notebook stores:

        stored_frame  = frame index in the stride-100 extxyz
        source_frame  = stored_frame * REACTIVE_SOURCE_STRIDE

    If the OPES file has one row per original trajectory frame, we align by
    source_frame. If this is not possible, we fall back to stored_frame.
    """
    out = n2_site_table.copy()

    out["opes_row"] = np.nan
    out["opes_time"] = np.nan
    out["opes_bias"] = np.nan
    out["opes_rct"] = np.nan
    out["opes_logw"] = np.nan
    out["opes_weight"] = np.nan

    for T, group in out.groupby("T"):
        T = int(T)

        if T not in opes_tables_by_temperature:
            print(f"{T} K: no OPES weights available; leaving weights as NaN.")
            continue

        opes = opes_tables_by_temperature[T]
        logw_all = compute_opes_logweights(
            opes,
            temperature=T,
            include_dwall=include_dwall,
        )

        group_index = group.index.to_numpy()

        source_rows = group["source_frame"].astype(int).to_numpy()
        stored_rows = group["stored_frame"].astype(int).to_numpy()

        if source_rows.max() < len(opes):
            rows = source_rows
            alignment = "source_frame"
        elif stored_rows.max() < len(opes):
            rows = stored_rows
            alignment = "stored_frame"
        else:
            raise IndexError(
                f"{T} K: cannot align OPES weights. "
                f"max source_frame={source_rows.max()}, max stored_frame={stored_rows.max()}, "
                f"OPES rows={len(opes)}."
            )

        print(f"{T} K: aligned OPES weights using {alignment}; OPES rows = {len(opes)}.")

        out.loc[group_index, "opes_row"] = rows
        out.loc[group_index, "opes_time"] = opes.iloc[rows]["time"].to_numpy(dtype=float) if "time" in opes.columns else np.nan
        out.loc[group_index, "opes_bias"] = opes.iloc[rows]["opes.bias"].to_numpy(dtype=float)
        out.loc[group_index, "opes_rct"] = opes.iloc[rows]["opes.rct"].to_numpy(dtype=float)
        out.loc[group_index, "opes_logw"] = logw_all[rows]
        out.loc[group_index, "opes_weight"] = np.exp(logw_all[rows])

    return out


def weighted_gaussian_kde_1d(
    values,
    weights,
    grid=np.linspace(0.0, 1.05, 400),
    bandwidth=None,
    value_range=(0.0, 1.05),
):
    """Weighted 1D Gaussian KDE normalized to unit integral."""
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)

    values = values[mask]
    weights = weights[mask]

    if len(values) == 0:
        raise ValueError("Cannot compute weighted KDE of an empty array.")

    weights = weights / weights.sum()

    if bandwidth is None:
        if len(values) == 1:
            bandwidth = 0.03 * (value_range[1] - value_range[0])
        else:
            # Effective sample size for weighted KDE.
            neff = 1.0 / np.sum(weights**2)
            std = np.sqrt(np.average((values - np.average(values, weights=weights))**2, weights=weights))
            q25, q75 = np.percentile(values, [25, 75])
            iqr_sigma = (q75 - q25) / 1.349 if q75 > q25 else std
            sigma = min(std, iqr_sigma) if std > 0 and iqr_sigma > 0 else max(std, iqr_sigma)

            if sigma <= 0 or not np.isfinite(sigma):
                sigma = 0.1 * (value_range[1] - value_range[0])

            bandwidth = 0.9 * sigma * neff ** (-1 / 5)

    if bandwidth <= 0 or not np.isfinite(bandwidth):
        bandwidth = 0.03 * (value_range[1] - value_range[0])

    grid = np.asarray(grid, dtype=float)

    z = (grid[:, None] - values[None, :]) / bandwidth
    density = np.exp(-0.5 * z**2) @ weights
    density /= bandwidth * np.sqrt(2 * np.pi)

    integral = np.trapezoid(density, grid)

    if integral > 0:
        density /= integral

    return grid, density


def plot_weighted_similarity_kde_by_temperature(
    table,
    value_col,
    weight_col="opes_weight",
    ax=None,
    title=None,
    label_suffix="",
    grid=np.linspace(0.0, 1.05, 400),
    bandwidth=None,
):
    """Plot OPES-weighted KDE curves grouped by temperature."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6.2, 3.8), dpi=140)

    plotted = False

    for T, group in table.groupby("T"):
        group = group[np.isfinite(group[weight_col])].copy()

        if len(group) == 0:
            print(f"{T} K: no finite OPES weights; skipping weighted KDE.")
            continue

        values = group[value_col].to_numpy()
        weights = group[weight_col].to_numpy()

        x, y = weighted_gaussian_kde_1d(
            values,
            weights,
            grid=grid,
            bandwidth=bandwidth,
            value_range=(grid.min(), grid.max()),
        )

        ax.plot(x, y, lw=1.8, label=f"{T} K{label_suffix}")
        plotted = True

    ax.axvline(THRESHOLD, ls="--", lw=1.0, label=r"$S=0.8$")
    ax.set_xlabel(r"$S(N_2,\chi_7)$")
    ax.set_ylabel("OPES-reweighted probability density")

    if title is not None:
        ax.set_title(title)

    if plotted:
        ax.legend(frameon=False)

    return ax


opes_tables = {}

for T, path in OPES_WEIGHT_FILES.items():
    if path.exists():
        opes_tables[T] = read_opes_colvar(path)
        print(f"{T} K OPES file loaded:", path)
        print("Rows:", len(opes_tables[T]))
        print("Columns:", list(opes_tables[T].columns))
    else:
        print(f"{T} K OPES file not found:", path)

n2_sites = attach_opes_weights_to_n2_sites(
    n2_sites,
    opes_tables,
    include_dwall=INCLUDE_DWALL_BIAS_IN_REWEIGHTING,
)

display(
    n2_sites[
        [
            "T",
            "frame",
            "stored_frame",
            "source_frame",
            "opes_row",
            "opes_time",
            "opes_bias",
            "opes_rct",
            "opes_weight",
            "d_NN_A",
            "abs_q_N2",
            "S_N2_chi7",
        ]
    ].head()
)

print("Available OPES-weighted rows:")
display(n2_sites.groupby("T")["opes_weight"].apply(lambda x: np.isfinite(x).sum()))

### 14.7c OPES-reweighted $S(N_2,\chi_7)$ distributions

The plots below repeat the reactive $S(N_2,\chi_7)$ KDE analysis using OPES weights.

With only the 300 K OPES file available, only the 300 K weighted curve is shown. Once the 700 K OPES file is provided, the same code will automatically include it.

In [ ]:
# -------------------------------------------------------------------------
# OPES-weighted KDE over all selected reactive frames
# -------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6.2, 3.8), dpi=140)

plot_weighted_similarity_kde_by_temperature(
    n2_sites,
    value_col="S_N2_chi7",
    weight_col="opes_weight",
    ax=ax,
    title=r"All selected reactive frames: OPES-reweighted KDE",
)

plt.show()


# -------------------------------------------------------------------------
# OPES-weighted KDE restricted to TS-like frames
# -------------------------------------------------------------------------

ts_data_weighted = n2_sites[n2_sites["ts_like"]].copy()

fig, ax = plt.subplots(figsize=(6.2, 3.8), dpi=140)

plot_weighted_similarity_kde_by_temperature(
    ts_data_weighted,
    value_col="S_N2_chi7",
    weight_col="opes_weight",
    ax=ax,
    title=(
        rf"TS-like frames only: OPES-reweighted KDE "
        rf"({TS_MIN_DNN:.1f} $\leq d_{{NN}} \leq$ {TS_MAX_DNN:.1f} Å)"
    ),
    label_suffix=" TS-like",
)

plt.show()


# -------------------------------------------------------------------------
# Weighted summary table
# -------------------------------------------------------------------------

def weighted_mean(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)

    if not np.any(mask):
        return np.nan

    return np.average(values[mask], weights=weights[mask])


weighted_summary_rows = []

for T, group in n2_sites.groupby("T"):
    weights = group["opes_weight"].to_numpy()
    weighted_summary_rows.append(
        {
            "T": T,
            "n_weighted": int(np.isfinite(weights).sum()),
            "weighted_mean_S_N2_chi7": weighted_mean(group["S_N2_chi7"], weights),
            "weighted_mean_d_NN_A": weighted_mean(group["d_NN_A"], weights),
            "weighted_mean_abs_q_N2": weighted_mean(group["abs_q_N2"], weights),
            "weighted_fraction_chi7_like": weighted_mean(group["active_site_below_N2"].astype(float), weights),
            "weighted_fraction_TS_like": weighted_mean(group["ts_like"].astype(float), weights),
        }
    )

weighted_reactive_summary = pd.DataFrame(weighted_summary_rows)
display(weighted_reactive_summary)

### 14.8 Committor analysis: identify the transition-state ensemble

In Bonati *et al.*, the final characterization of the active site at the transition state is not based only on a geometric window in the N--N distance. The paper selects a set of candidate configurations around the barrier and then runs short unbiased trajectories from each configuration. A configuration is classified as part of the transition-state ensemble when its committor probability is close to 0.5.

The folders provided here contain committor runs for snapshots at 300 K and 700 K. Each folder has the structure:

```text
frame-XXXXX/
  data.lammps
  plumed.dat
  results.txt
```

The `plumed.dat` file defines two basins using the N--N distance:

```text
BASIN 1: 1.0 <= dNN <= 1.4 Å   molecular N2 basin
BASIN 2: 2.0 <= dNN <= 4.0 Å   dissociated 2N basin
```

Therefore, for each snapshot we estimate:

\[
p_\mathrm{diss} = \frac{N(\mathrm{BASIN\ 2})}{N(\mathrm{BASIN\ 1}) + N(\mathrm{BASIN\ 2})}.
\]

We classify a snapshot as committor-TS-like when:

\[
0.25 \le p_\mathrm{diss} \le 0.75.
\]

This reproduces the logic used in the paper more closely than the earlier purely geometric filter \(1.6 \le d_{\mathrm{NN}} \le 1.8\ \mathrm{\AA}\). The geometric filter remains useful for a quick tutorial analysis, while the committor filter is the more rigorous transition-state definition.

In [ ]:
# -------------------------------------------------------------------------
# 15.8 Parse committor snapshots and identify TS-like snapshots
# -------------------------------------------------------------------------

COMM_DIR = DATA / "committor"
COMM_ARCHIVES = {
    300: COMM_DIR / "snapshots_300K.tar.gz",
    700: COMM_DIR / "snapshots_700K.tar.gz",
}
COMM_EXTRACTED = COMM_DIR / "extracted"
COMM_EXTRACTED.mkdir(parents=True, exist_ok=True)

COMM_TS_MIN = 0.25
COMM_TS_MAX = 0.75


def extract_committor_archive(temp, archive_path):
    """Extract one committor snapshot archive if needed."""
    archive_path = Path(archive_path)
    if not archive_path.exists():
        raise FileNotFoundError(archive_path)

    target = COMM_EXTRACTED / f"snapshots_{temp}K"
    if target.exists():
        return target

    tmp = COMM_EXTRACTED / f"tmp_{temp}K"
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True)

    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(tmp)

    extracted_snapshots = tmp / "snapshots"
    if not extracted_snapshots.exists():
        raise RuntimeError(f"Archive {archive_path} did not contain a 'snapshots' folder.")

    extracted_snapshots.rename(target)
    shutil.rmtree(tmp)
    return target


def parse_committor_snapshot_folder(temp, snapshot_root):
    """Parse all frame-* folders from one committor snapshot directory."""
    rows = []

    for frame_dir in sorted(Path(snapshot_root).glob("frame-*"), key=lambda p: int(p.name.split("-")[1])):
        frame_id = int(frame_dir.name.split("-")[1])
        result_file = frame_dir / "results.txt"
        data_file = frame_dir / "data.lammps"
        plumed_file = frame_dir / "plumed.dat"

        if result_file.exists():
            text = result_file.read_text(errors="ignore")
            n_basin1 = text.count("COMMITTED TO BASIN 1")
            n_basin2 = text.count("COMMITTED TO BASIN 2")
        else:
            n_basin1 = 0
            n_basin2 = 0

        n_total = n_basin1 + n_basin2
        p_n2 = n_basin1 / n_total if n_total > 0 else np.nan
        p_diss = n_basin2 / n_total if n_total > 0 else np.nan

        rows.append(
            {
                "T": temp,
                "frame_id": frame_id,
                "snapshot_dir": str(frame_dir),
                "data_file": str(data_file),
                "plumed_file": str(plumed_file),
                "result_file": str(result_file),
                "has_results": result_file.exists(),
                "n_basin1_N2": n_basin1,
                "n_basin2_2N": n_basin2,
                "n_trials": n_total,
                "p_N2": p_n2,
                "p_diss": p_diss,
                "committor_ts": bool(np.isfinite(p_diss) and COMM_TS_MIN <= p_diss <= COMM_TS_MAX),
            }
        )

    return pd.DataFrame(rows)


committor_tables = []
committor_snapshot_roots = {}

for T, archive in COMM_ARCHIVES.items():
    root = extract_committor_archive(T, archive)
    committor_snapshot_roots[T] = root
    table = parse_committor_snapshot_folder(T, root)
    committor_tables.append(table)

committor_table = pd.concat(committor_tables, ignore_index=True)

display(committor_table.head())

print("Committor snapshot summary:")
display(
    committor_table
    .groupby("T")
    .agg(
        n_snapshots=("frame_id", "count"),
        n_with_results=("has_results", "sum"),
        n_committor_ts=("committor_ts", "sum"),
        mean_trials=("n_trials", "mean"),
    )
)

print("Committor probability summary for snapshots with results:")
display(
    committor_table[committor_table["has_results"]]
    .groupby("T")["p_diss"]
    .describe()
)

fig, ax = plt.subplots(figsize=(6.2, 3.8), dpi=140)

for T, group in committor_table[committor_table["has_results"]].groupby("T"):
    ax.hist(
        group["p_diss"],
        bins=np.linspace(0, 1, 21),
        histtype="step",
        lw=1.8,
        label=f"{T} K",
    )

ax.axvspan(COMM_TS_MIN, COMM_TS_MAX, alpha=0.15, label="committor TS window")
ax.set_xlabel(r"$p_\mathrm{diss}$")
ax.set_ylabel("Number of snapshots")
ax.set_title("Committor probabilities from short unbiased trajectories")
ax.legend(frameon=False)
plt.show()

### 14.9 Active-site analysis on committor-selected TS snapshots

We now repeat the local active-site analysis on the **committor-selected transition-state ensemble**.

This section intentionally uses the same tools as the rest of the tutorial:

```text
committor TS data.lammps snapshots
  -> ASE reads the structures
  -> OVITO recomputes the upper-surface atoms
  -> PLUMED computes S(chi, chi7)
  -> Python selects the Fe atom below/around N2
  -> KDE of S(N2, chi7) for the committor TS ensemble
```

Compared with the previous TS-like geometric window, this is closer to the analysis in Fig. 7 of the paper. The first plot in this section is unweighted and is meant as a tutorial-level structural comparison of the selected TS snapshots. The following section, 15.9b, attaches OPES weights to the committor-selected snapshots and repeats the distribution analysis in reweighted form.

Cached committor surface files are validated not only for ASE readability, but also for the presence of the `SelectionSurface` array. If a stale cached file is readable but lacks `SelectionSurface`, the notebook deletes it and regenerates the OVITO output.

If `SelectionSurface` is missing but OVITO's standard `Selection` property is present, the notebook now repairs the trajectory by reconstructing the upper-surface mask from `Selection` and the slab mid-plane. This handles cached files produced by older notebook versions or by OVITO/ASE combinations that do not preserve the custom `SelectionSurface` property during conversion.

Version v34 also fixes a more subtle issue: even if `SelectionSurface` is correctly attached to the ASE frames in memory, some ASE versions do not write custom arrays unless they are explicitly listed in `columns`. Therefore, the committor surface cache is now written with explicit columns and immediately read back to verify that `SelectionSurface` survived.



In [ ]:
# -------------------------------------------------------------------------
# 14.9 Active-site analysis on committor-selected TS snapshots
# -------------------------------------------------------------------------

COMM_FORCE_RECOMPUTE = False
COMM_DELETE_SURFACE_CACHE = False  # set True once if old cached files still cause problems
COMM_LOCAL_N2_CUTOFF_A = 3.0


def read_committor_lammps_snapshot(path):
    """Read one committor LAMMPS data snapshot as an ASE Atoms object."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    atoms = read(
        path,
        format="lammps-data",
        style="atomic",
        Z_of_type={1: 26, 2: 7},
    )
    atoms.pbc = [True, True, False]
    return atoms


def load_committor_ts_snapshots(comm_table, temp):
    """Load committor-selected TS snapshots for one temperature."""
    rows = (
        comm_table[
            (comm_table["T"] == temp) &
            (comm_table["committor_ts"]) &
            (comm_table["has_results"])
        ]
        .sort_values("frame_id")
        .reset_index(drop=True)
    )

    traj = []
    for _, row in rows.iterrows():
        atoms = read_committor_lammps_snapshot(row["data_file"])
        atoms.info["T_K"] = int(temp)
        atoms.info["committor_frame_id"] = int(row["frame_id"])
        atoms.info["p_diss"] = float(row["p_diss"])
        atoms.info["n_trials"] = int(row["n_trials"])
        traj.append(atoms)

    return traj, rows


committor_ts_raw_trajs = {}
committor_ts_metadata = {}
committor_ts_trajs = {}

if COMM_DELETE_SURFACE_CACHE:
    for cached in COMM_DIR.glob("committor_TS_*_surface.extxyz"):
        print("Deleting old committor surface cache:", cached)
        cached.unlink()
    for cached in COMM_DIR.glob("committor_TS_*_plumed*"):
        print("Deleting old committor PLUMED cache:", cached)
        if cached.is_dir():
            shutil.rmtree(cached)
        else:
            cached.unlink()

for T in [300, 700]:
    print(f"\nPreparing committor TS snapshots at {T} K")

    raw_traj, meta = load_committor_ts_snapshots(committor_table, T)
    committor_ts_raw_trajs[T] = raw_traj
    committor_ts_metadata[T] = meta

    print(f"Loaded committor TS snapshots: {len(raw_traj)}")

    input_xyz = COMM_DIR / f"committor_TS_{T}K_input.extxyz"
    surface_xyz = COMM_DIR / f"committor_TS_{T}K_surface.extxyz"

    if COMM_FORCE_RECOMPUTE or not input_xyz.exists():
        write(input_xyz, raw_traj, format="extxyz")
        print("Wrote:", input_xyz)
    else:
        print("Using cached:", input_xyz)

    cache_ok = False
    if surface_xyz.exists() and not COMM_FORCE_RECOMPUTE:
        try:
            load_surface_trajectory(surface_xyz)
            cache_ok = True
            print(f"{T} K: using cached surface trajectory:", surface_xyz)
        except Exception as exc:
            print(f"{T} K: cached surface trajectory is not usable ({type(exc).__name__}); regenerating.")

    if not cache_ok:
        if surface_xyz.exists():
            print(f"{T} K: deleting invalid cached surface trajectory:", surface_xyz)
            surface_xyz.unlink()

        print(f"{T} K: recomputing surface selection for committor TS snapshots with OVITO...")
        add_upper_surface_selection_with_ovito(
            input_file=input_xyz,
            output_file=surface_xyz,
            alpha_radius=2.0,
        )

    traj = load_surface_trajectory(surface_xyz)
    committor_ts_trajs[T] = traj

    if len(traj) != len(meta):
        raise RuntimeError(f"{T} K: surface trajectory length {len(traj)} != metadata length {len(meta)}")

committor_plumed_values = {}

for T, traj in committor_ts_trajs.items():
    print(f"\nRunning/reading PLUMED for committor TS snapshots at {T} K")

    work_comm = PLUMED_DIR / f"committor_TS_{T}K"
    work_comm.mkdir(parents=True, exist_ok=True)
    ref_comm = write_plumed_reference_pdb(ref_vecs, work_comm / "chi7_reference.pdb")
    plumed_comm = write_plumed_envsim_input(
        work_comm / "plumed.dat",
        ref_comm,
        n_atoms=768,
        sigma=SIGMA_BY_T[T],
        threshold=THRESHOLD,
    )

    xyz_comm = work_comm / f"committor_TS_{T}K_for_plumed.xyz"

    if COMM_FORCE_RECOMPUTE or not xyz_comm.exists():
        write_xyz_for_plumed_with_box(traj, xyz_comm)

    colvar_comm = work_comm / "COLVAR"
    multicolvar_comm = work_comm / "envsim_multicolvar.xyz"

    if COMM_FORCE_RECOMPUTE or not (colvar_comm.exists() and multicolvar_comm.exists()):
        run_plumed_driver(work_comm, xyz_comm, plumed_comm)
    else:
        print("Using cached PLUMED outputs.")

    raw_s = read_plumed_multicolvar_xyz(multicolvar_comm, T)
    committor_plumed_values[T] = assign_plumed_rows_to_fe_atoms(raw_s, traj, T)


def committor_ts_n2_descriptors(traj, plumed_table, metadata, temp, cutoff_to_n2=COMM_LOCAL_N2_CUTOFF_A):
    """Build a descriptor table for committor-selected TS snapshots."""
    rows = []
    s_lookup = plumed_table.set_index(["frame", "atom_id"])["S_plumed"].to_dict()

    for frame, atoms in enumerate(traj):
        meta = metadata.iloc[frame]

        symbols = np.array(atoms.get_chemical_symbols())
        positions = atoms.get_positions()
        n_ids = np.where(symbols == "N")[0]
        fe_ids = np.where(symbols == "Fe")[0]

        if len(n_ids) != 2:
            raise ValueError(f"Expected two N atoms in frame {frame}, found {len(n_ids)}")

        d_NN = atoms.get_distance(n_ids[0], n_ids[1], mic=True)
        distances = atoms.get_all_distances(mic=True)

        near_n2 = (distances[n_ids[:, None], fe_ids] < cutoff_to_n2).any(axis=0)
        candidate_fe = fe_ids[near_n2]

        if len(candidate_fe) == 0:
            site_atom_id = np.nan
            s_n2_chi7 = np.nan
            site_is_surface = False
        else:
            site_atom_id = int(candidate_fe[np.argmin(positions[candidate_fe, 2])])
            s_n2_chi7 = float(s_lookup.get((frame, site_atom_id), np.nan))
            site_is_surface = bool(atoms.arrays["SelectionSurface"][site_atom_id])

        rows.append(
            {
                "T": int(temp),
                "frame": int(frame),
                "committor_frame_id": int(meta["frame_id"]),
                "p_diss": float(meta["p_diss"]),
                "p_N2": float(meta["p_N2"]),
                "n_trials": int(meta["n_trials"]),
                "d_NN_A": float(d_NN),
                "site_atom_id": site_atom_id,
                "site_is_surface": site_is_surface,
                "S_N2_chi7": s_n2_chi7,
                "active_site_below_N2": bool(s_n2_chi7 >= THRESHOLD) if np.isfinite(s_n2_chi7) else False,
            }
        )

    return pd.DataFrame(rows)


committor_ts_site_tables = []
for T, traj in committor_ts_trajs.items():
    table = committor_ts_n2_descriptors(
        traj,
        committor_plumed_values[T],
        committor_ts_metadata[T],
        temp=T,
    )
    committor_ts_site_tables.append(table)

committor_ts_sites = pd.concat(committor_ts_site_tables, ignore_index=True)

display(committor_ts_sites.head())

print("Committor TS active-site summary:")
display(
    committor_ts_sites
    .groupby("T")
    .agg(
        n_TS=("frame", "count"),
        mean_p_diss=("p_diss", "mean"),
        mean_d_NN_A=("d_NN_A", "mean"),
        mean_S_N2_chi7=("S_N2_chi7", "mean"),
        fraction_chi7_like=("active_site_below_N2", "mean"),
    )
)

# Smooth KDE distribution of S(N2, chi7) for the committor TS ensemble.
fig, ax = plt.subplots(figsize=(6.2, 3.8), dpi=140)

plot_similarity_kde_by_temperature(
    committor_ts_sites,
    value_col="S_N2_chi7",
    ax=ax,
    title=r"Committor-selected TS snapshots: $S(N_2,\chi_7)$",
    label_suffix=" committor TS",
)

ax.set_xlabel(r"$S(N_2,\chi_7)$")
plt.show()

# Compare with the earlier geometric TS-like selection.
fig, ax = plt.subplots(figsize=(6.2, 3.8), dpi=140)

if "ts_data" in globals() and len(ts_data) > 0:
    plot_similarity_kde_by_temperature(
        ts_data,
        value_col="S_N2_chi7",
        ax=ax,
        title=r"Geometric TS-like vs committor TS snapshots",
        label_suffix=" geom. TS",
    )

plot_similarity_kde_by_temperature(
    committor_ts_sites,
    value_col="S_N2_chi7",
    ax=ax,
    title=r"Geometric TS-like vs committor TS snapshots",
    label_suffix=" committor TS",
)

ax.set_xlabel(r"$S(N_2,\chi_7)$")
plt.show()


### 14.9b OPES-reweighted committor-selected TS snapshots

The committor trajectories themselves are short unbiased trajectories, but the **initial configurations** used for committor analysis were extracted from the OPES-biased reactive trajectory.

Therefore, when we use the selected TS snapshots to build a probability distribution of \(S(N_2,\chi_7)\), we should attach the OPES statistical weight of the parent snapshot.

This section aligns each `frame-XXXXX` committor snapshot with the corresponding row of the OPES/COLVAR file:

```text
committor_frame_id  ->  OPES row
```

and computes weighted KDEs and weighted summary statistics for the committor-selected TS ensemble.

Important distinction:

```text
committor analysis  -> classifies whether a configuration is TS-like
OPES reweighting    -> corrects the statistical probability of selected configurations
```

The weighted curves are the ones that should be compared most directly with the reweighted distributions reported in the paper.

In [ ]:
# -------------------------------------------------------------------------
# 15.9b OPES reweighting for committor-selected TS snapshots
# -------------------------------------------------------------------------

if "opes_tables" not in globals():
    raise RuntimeError(
        "Run section 15.7b first. It defines opes_tables and the OPES reweighting helpers."
    )

if "committor_ts_sites" not in globals():
    raise RuntimeError("Run section 15.9 first. It defines committor_ts_sites.")


def attach_opes_weights_to_committor_ts_sites(
    comm_sites,
    opes_tables_by_temperature,
    include_dwall=False,
):
    """Attach OPES weights to committor-selected TS snapshots.

    The committor folders are named frame-XXXXX. These numbers correspond to
    the parent frame in the OPES trajectory, so we use committor_frame_id as the
    OPES row index.
    """
    out = comm_sites.copy()

    out["comm_opes_row"] = np.nan
    out["comm_opes_time"] = np.nan
    out["comm_opes_bias"] = np.nan
    out["comm_opes_rct"] = np.nan
    out["comm_opes_logw"] = np.nan
    out["comm_opes_weight"] = np.nan

    for T, group in out.groupby("T"):
        T = int(T)

        if T not in opes_tables_by_temperature:
            print(f"{T} K: no OPES weights available for committor TS snapshots.")
            continue

        opes = opes_tables_by_temperature[T]
        logw_all = compute_opes_logweights(
            opes,
            temperature=T,
            include_dwall=include_dwall,
        )

        rows = group["committor_frame_id"].astype(int).to_numpy()

        if rows.max() >= len(opes):
            raise IndexError(
                f"{T} K: cannot align committor frame IDs with OPES rows. "
                f"max committor_frame_id={rows.max()}, OPES rows={len(opes)}."
            )

        group_index = group.index.to_numpy()

        print(
            f"{T} K: aligned {len(group)} committor TS snapshots "
            f"using committor_frame_id -> OPES row; OPES rows = {len(opes)}."
        )

        out.loc[group_index, "comm_opes_row"] = rows
        out.loc[group_index, "comm_opes_time"] = (
            opes.iloc[rows]["time"].to_numpy(dtype=float)
            if "time" in opes.columns
            else np.nan
        )
        out.loc[group_index, "comm_opes_bias"] = opes.iloc[rows]["opes.bias"].to_numpy(dtype=float)
        out.loc[group_index, "comm_opes_rct"] = opes.iloc[rows]["opes.rct"].to_numpy(dtype=float)
        out.loc[group_index, "comm_opes_logw"] = logw_all[rows]
        out.loc[group_index, "comm_opes_weight"] = np.exp(logw_all[rows])

    return out


def effective_sample_size(weights):
    """Compute the Kish effective sample size."""
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(weights) & (weights > 0)
    weights = weights[mask]

    if len(weights) == 0:
        return np.nan

    weights = weights / weights.sum()
    return 1.0 / np.sum(weights**2)


committor_ts_sites = attach_opes_weights_to_committor_ts_sites(
    committor_ts_sites,
    opes_tables,
    include_dwall=INCLUDE_DWALL_BIAS_IN_REWEIGHTING,
)

print("Committor TS snapshots with OPES weights:")
display(
    committor_ts_sites[
        [
            "T",
            "frame",
            "committor_frame_id",
            "p_diss",
            "comm_opes_row",
            "comm_opes_time",
            "comm_opes_bias",
            "comm_opes_rct",
            "comm_opes_weight",
            "S_N2_chi7",
            "active_site_below_N2",
        ]
    ].head()
)

print("OPES-weight availability for committor TS snapshots:")
display(
    committor_ts_sites
    .groupby("T")["comm_opes_weight"]
    .apply(lambda x: np.isfinite(x).sum())
)

print("Effective sample size after OPES reweighting:")
display(
    committor_ts_sites
    .groupby("T")["comm_opes_weight"]
    .apply(effective_sample_size)
    .rename("N_eff")
)


# -------------------------------------------------------------------------
# Weighted KDE of S(N2, chi7) for committor-selected TS snapshots
# -------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6.2, 3.8), dpi=140)

plot_weighted_similarity_kde_by_temperature(
    committor_ts_sites,
    value_col="S_N2_chi7",
    weight_col="comm_opes_weight",
    ax=ax,
    title=r"Committor-selected TS snapshots: OPES-reweighted $S(N_2,\chi_7)$",
    label_suffix=" committor TS",
)

ax.set_xlabel(r"$S(N_2,\chi_7)$")
plt.show()


# -------------------------------------------------------------------------
# Unweighted vs weighted comparison
# -------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6.2, 3.8), dpi=140)

# Unweighted curves as dashed lines.
for T, group in committor_ts_sites.groupby("T"):
    values = group["S_N2_chi7"].dropna().to_numpy()

    if len(values) == 0:
        continue

    x, y = gaussian_kde_1d(values, grid=np.linspace(0.0, 1.05, 400))
    ax.plot(x, y, ls="--", lw=1.3, label=f"{T} K unweighted")

# Weighted curves as solid lines.
for T, group in committor_ts_sites.groupby("T"):
    group = group[np.isfinite(group["comm_opes_weight"])].copy()

    if len(group) == 0:
        continue

    x, y = weighted_gaussian_kde_1d(
        group["S_N2_chi7"].to_numpy(),
        group["comm_opes_weight"].to_numpy(),
        grid=np.linspace(0.0, 1.05, 400),
    )
    ax.plot(x, y, lw=2.0, label=f"{T} K OPES-weighted")

ax.axvline(THRESHOLD, ls=":", lw=1.0, label=r"$S=0.8$")
ax.set_xlabel(r"$S(N_2,\chi_7)$")
ax.set_ylabel("Probability density")
ax.set_title("Effect of OPES reweighting on the committor TS distribution")
ax.legend(frameon=False, fontsize=8)
plt.show()


# -------------------------------------------------------------------------
# Weighted summary table
# -------------------------------------------------------------------------

comm_weighted_summary_rows = []

for T, group in committor_ts_sites.groupby("T"):
    weights = group["comm_opes_weight"].to_numpy()

    comm_weighted_summary_rows.append(
        {
            "T": int(T),
            "n_TS": len(group),
            "N_eff": effective_sample_size(weights),
            "weighted_mean_p_diss": weighted_mean(group["p_diss"], weights),
            "weighted_mean_d_NN_A": weighted_mean(group["d_NN_A"], weights),
            "weighted_mean_S_N2_chi7": weighted_mean(group["S_N2_chi7"], weights),
            "weighted_fraction_chi7_like": weighted_mean(
                group["active_site_below_N2"].astype(float),
                weights,
            ),
        }
    )

committor_weighted_summary = pd.DataFrame(comm_weighted_summary_rows)

display(committor_weighted_summary)



### 14.9c Optional diagnostic: parent reactive frames vs isolated committor snapshots

The previous committor analysis recomputed \(S(N_2,\chi_7)\) directly on the isolated `data.lammps` snapshots.

Here we test a closer strategy to the original workflow:

```text
committor frame id
  -> corresponding parent frame in the reactive OPES trajectory
  -> OVITO/PLUMED active-site analysis on that parent reactive frame
  -> OPES-weighted KDE of S(N2, chi7)
```

This is useful because the committor `data.lammps` files are isolated snapshots, while the original post-processing pipeline computes environment similarity on trajectory frames after the OVITO/PLUMED workflow.

Implementation detail for the tutorial data:

- the reactive trajectory included in this package is already **stride-100** with respect to the original OPES trajectory;
- therefore, if an exact parent frame is not available, we use the nearest stored stride-100 parent frame and report the frame mismatch;
- if you later provide a full OVITO-processed/smoothed reactive trajectory with `EnvSimilarity`, this section can be adapted to read that file directly.


This is an optional diagnostic section. It is useful when comparing the tutorial output with the published paper, but it is not required to understand the core committor workflow.


In [ ]:
# -------------------------------------------------------------------------
# 15.9c Parent-reactive-frame test for committor TS snapshots
# -------------------------------------------------------------------------

FORCE_RECOMPUTE_COMM_PARENT = False

# The included reactive N2 trajectories are stride-100 relative to the original
# OPES trajectory. Committor frame IDs refer to original trajectory frame IDs.
COMM_PARENT_SOURCE_STRIDE = REACTIVE_SOURCE_STRIDE

# Optional stricter committor window, closer to "p_c ≈ 50%".
COMM_STRICT_TS_MIN = 0.45
COMM_STRICT_TS_MAX = 0.55


def nearest_parent_stored_frame(source_frame, source_stride=COMM_PARENT_SOURCE_STRIDE):
    """Map an original-frame ID to the nearest stored frame of a strided trajectory."""
    stored = int(np.rint(float(source_frame) / float(source_stride)))
    nearest_source = stored * int(source_stride)
    delta = int(nearest_source - int(source_frame))
    return stored, nearest_source, delta


def load_reactive_frames_by_stored_indices(path, stored_indices):
    """Read selected stored frames from an extxyz trajectory.

    Duplicates in stored_indices are preserved in the returned list, so the
    returned trajectory remains one-to-one with the committor metadata table.
    """
    path = Path(path)
    stored_indices = [int(i) for i in stored_indices]
    unique_indices = sorted(set(stored_indices))

    if len(unique_indices) == 0:
        return []

    max_index = max(unique_indices)
    frame_cache = {}

    for iframe, atoms in enumerate(iread(path, index=":", format="extxyz")):
        if iframe in unique_indices:
            frame_cache[iframe] = atoms.copy()

        if iframe >= max_index:
            break

    missing = [i for i in unique_indices if i not in frame_cache]

    if missing:
        raise IndexError(
            f"Could not read stored frames {missing[:10]} from {path}. "
            f"Highest requested index was {max_index}."
        )

    return [frame_cache[i].copy() for i in stored_indices]






def attach_surface_and_active_flags(
    plumed_table,
    ase_traj,
    temp=None,
    threshold=THRESHOLD,
):
    """Match PLUMED rows to ASE atoms and attach surface/active-site flags.

    This helper is used in the parent-reactive-frame committor test.
    It uses the same row-order assignment introduced in section 7.

    Parameters
    ----------
    plumed_table
        Output of `read_plumed_multicolvar_xyz()`, i.e. one S_plumed value per Fe atom.
    ase_traj
        ASE trajectory on which PLUMED was run. It must contain the
        SelectionSurface array.
    temp
        Temperature label. If None, it is inferred from plumed_table['T'].
    threshold
        Similarity threshold used to mark chi7-like environments.
    """
    if temp is None:
        if "T" not in plumed_table.columns:
            raise ValueError("temp is None and plumed_table has no 'T' column.")
        unique_T = pd.unique(plumed_table["T"])
        if len(unique_T) != 1:
            raise ValueError(f"Cannot infer a unique temperature from T values: {unique_T}")
        temp = int(unique_T[0])

    matched = assign_plumed_rows_to_fe_atoms(
        plumed_table,
        ase_traj,
        temp,
    )

    surface_lookup = {}
    for iframe, atoms in enumerate(ase_traj):
        if "SelectionSurface" not in atoms.arrays:
            raise KeyError(
                "SelectionSurface is missing from the ASE trajectory used for "
                "parent reactive frame analysis."
            )
        surface = atoms.arrays["SelectionSurface"].astype(bool)
        for atom_id in range(len(atoms)):
            surface_lookup[(iframe, atom_id)] = bool(surface[atom_id])

    matched["SelectionSurface"] = [
        surface_lookup.get((int(row.frame), int(row.atom_id)), False)
        for row in matched.itertuples(index=False)
    ]
    matched["active_chi7_like"] = (
        matched["SelectionSurface"].astype(bool) &
        (matched["S_plumed"] >= threshold)
    )

    return matched


def load_parent_reactive_frames_for_committor_ts(temp, comm_sites):
    """Load nearest parent reactive frames for committor-selected TS snapshots."""
    rows = (
        comm_sites[comm_sites["T"] == temp]
        .sort_values("frame")
        .reset_index(drop=True)
    )

    if len(rows) == 0:
        return [], rows

    mapping = [
        nearest_parent_stored_frame(frame_id)
        for frame_id in rows["committor_frame_id"].astype(int).to_numpy()
    ]

    parent_stored = [m[0] for m in mapping]
    parent_source = [m[1] for m in mapping]
    parent_delta = [m[2] for m in mapping]

    parent_traj = load_reactive_frames_by_stored_indices(
        REACTIVE_N2_TRAJECTORIES[temp],
        parent_stored,
    )

    rows = rows.copy()
    rows["parent_stored_frame"] = parent_stored
    rows["parent_source_frame"] = parent_source
    rows["parent_frame_delta"] = parent_delta

    for atoms, (_, row) in zip(parent_traj, rows.iterrows()):
        atoms.info["T_K"] = int(temp)
        atoms.info["committor_frame_id"] = int(row["committor_frame_id"])
        atoms.info["parent_stored_frame"] = int(row["parent_stored_frame"])
        atoms.info["parent_source_frame"] = int(row["parent_source_frame"])
        atoms.info["parent_frame_delta"] = int(row["parent_frame_delta"])

    return parent_traj, rows


committor_parent_raw_trajs = {}
committor_parent_trajs = {}
committor_parent_metadata = {}
committor_parent_plumed_values = {}

for T in [300, 700]:
    print(f"\nPreparing parent reactive frames for committor TS snapshots at {T} K")

    parent_raw_traj, parent_meta = load_parent_reactive_frames_for_committor_ts(
        T,
        committor_ts_sites,
    )

    committor_parent_raw_trajs[T] = parent_raw_traj
    committor_parent_metadata[T] = parent_meta

    print("Parent frames loaded:", len(parent_raw_traj))

    if len(parent_meta) > 0:
        print(
            "Frame mismatch relative to exact committor frame, in original MD steps:",
            {
                "min": int(parent_meta["parent_frame_delta"].min()),
                "max": int(parent_meta["parent_frame_delta"].max()),
                "mean_abs": float(parent_meta["parent_frame_delta"].abs().mean()),
            },
        )

    input_xyz = COMM_DIR / f"committor_parent_reactive_{T}K_input.extxyz"
    surface_xyz = COMM_DIR / f"committor_parent_reactive_{T}K_surface.extxyz"

    if FORCE_RECOMPUTE_COMM_PARENT or not input_xyz.exists():
        write(input_xyz, parent_raw_traj, format="extxyz")
        print("Wrote:", input_xyz)
    else:
        print("Using cached:", input_xyz)

    regenerate_surface = FORCE_RECOMPUTE_COMM_PARENT

    if surface_xyz.exists() and not FORCE_RECOMPUTE_COMM_PARENT:
        try:
            load_surface_trajectory(surface_xyz)
            regenerate_surface = False
            print("Using cached parent surface trajectory:", surface_xyz)
        except Exception as exc:
            print(f"Cached parent surface trajectory is not usable ({type(exc).__name__}); regenerating.")
            regenerate_surface = True
    else:
        regenerate_surface = True

    if regenerate_surface:
        print(f"{T} K: recomputing OVITO surface selection on parent reactive frames...")
        add_upper_surface_selection_with_ovito(
            input_file=input_xyz,
            output_file=surface_xyz,
            alpha_radius=2.0,
        )

    parent_surface_traj = load_surface_trajectory(surface_xyz)

    committor_parent_trajs[T] = parent_surface_traj

    workdir = PLUMED_DIR / f"committor_parent_reactive_{T}K"
    workdir.mkdir(parents=True, exist_ok=True)
    ref_file = write_plumed_reference_pdb(ref_vecs, workdir / "chi7_reference.pdb")
    plumed_file = write_plumed_envsim_input(
        workdir / "plumed.dat",
        ref_file,
        n_atoms=768,
        sigma=SIGMA_BY_T[T],
        threshold=THRESHOLD,
    )
    xyz_for_plumed = workdir / f"committor_parent_reactive_{T}K_for_plumed.xyz"

    if FORCE_RECOMPUTE_COMM_PARENT or not xyz_for_plumed.exists():
        write_xyz_for_plumed_with_box(parent_surface_traj, xyz_for_plumed)
        print("Wrote PLUMED XYZ:", xyz_for_plumed)
    else:
        print("Using cached PLUMED XYZ:", xyz_for_plumed)

    if FORCE_RECOMPUTE_COMM_PARENT or not ((workdir / "COLVAR").exists() and (workdir / "envsim_multicolvar.xyz").exists()):
        print(f"{T} K: running PLUMED driver on parent reactive frames...")
        colvar_parent, multicolvar_parent = run_plumed_driver(
            workdir,
            xyz_for_plumed,
            plumed_file,
        )
    else:
        print(f"{T} K: using cached PLUMED outputs for parent reactive frames.")
        colvar_parent = workdir / "COLVAR"
        multicolvar_parent = workdir / "envsim_multicolvar.xyz"

    raw_s_parent = read_plumed_multicolvar_xyz(multicolvar_parent, T)
    committor_parent_plumed_values[T] = attach_surface_and_active_flags(
        raw_s_parent,
        parent_surface_traj,
        temp=T,
        threshold=THRESHOLD,
    )


# Build descriptor table using the parent reactive frames.
committor_parent_site_tables = []

for T, traj in committor_parent_trajs.items():
    meta = committor_parent_metadata[T]

    parent_table = reactive_n2_descriptors(
        traj,
        committor_parent_plumed_values[T],
        temp=T,
        stored_frame_indices=meta["parent_stored_frame"].astype(int).tolist(),
        source_frames=meta["parent_source_frame"].astype(int).tolist(),
        cutoff_to_n2=COMM_LOCAL_N2_CUTOFF_A,
    )

    # Keep the original committor metadata and OPES weights.
    parent_table["committor_frame_id"] = meta["committor_frame_id"].astype(int).to_numpy()
    parent_table["parent_frame_delta"] = meta["parent_frame_delta"].astype(int).to_numpy()
    parent_table["p_diss"] = meta["p_diss"].astype(float).to_numpy()
    parent_table["p_N2"] = meta["p_N2"].astype(float).to_numpy()
    parent_table["n_trials"] = meta["n_trials"].astype(int).to_numpy()

    for col in [
        "comm_opes_row",
        "comm_opes_time",
        "comm_opes_bias",
        "comm_opes_rct",
        "comm_opes_logw",
        "comm_opes_weight",
    ]:
        if col in meta.columns:
            parent_table[col] = meta[col].to_numpy()

    committor_parent_site_tables.append(parent_table)

committor_parent_ts_sites = pd.concat(committor_parent_site_tables, ignore_index=True)

display(
    committor_parent_ts_sites[
        [
            "T",
            "committor_frame_id",
            "stored_frame",
            "source_frame",
            "parent_frame_delta",
            "p_diss",
            "d_NN_A",
            "S_N2_chi7",
            "active_site_below_N2",
            "comm_opes_weight",
        ]
    ].head()
)

print("Parent-reactive-frame TS summary:")
display(
    committor_parent_ts_sites
    .groupby("T")
    .agg(
        n_TS=("frame", "count"),
        mean_abs_parent_delta=("parent_frame_delta", lambda x: np.mean(np.abs(x))),
        mean_p_diss=("p_diss", "mean"),
        mean_d_NN_A=("d_NN_A", "mean"),
        mean_S_N2_chi7=("S_N2_chi7", "mean"),
        fraction_chi7_like=("active_site_below_N2", "mean"),
    )
)


# -------------------------------------------------------------------------
# OPES-weighted KDE using parent reactive frames
# -------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6.2, 3.8), dpi=140)

plot_weighted_similarity_kde_by_temperature(
    committor_parent_ts_sites,
    value_col="S_N2_chi7",
    weight_col="comm_opes_weight",
    ax=ax,
    title=r"Parent reactive frames for committor TS: OPES-reweighted $S(N_2,\chi_7)$",
    label_suffix=" parent",
)

ax.set_xlabel(r"$S(N_2,\chi_7)$")
plt.show()


# -------------------------------------------------------------------------
# Compare isolated committor snapshots vs parent reactive frames
# -------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6.2, 3.8), dpi=140)

for T, group in committor_ts_sites.groupby("T"):
    group = group[np.isfinite(group["comm_opes_weight"])].copy()

    if len(group) == 0:
        continue

    x, y = weighted_gaussian_kde_1d(
        group["S_N2_chi7"].to_numpy(),
        group["comm_opes_weight"].to_numpy(),
        grid=np.linspace(0.0, 1.05, 400),
    )

    ax.plot(x, y, ls="--", lw=1.4, label=f"{T} K isolated snapshot")

for T, group in committor_parent_ts_sites.groupby("T"):
    group = group[np.isfinite(group["comm_opes_weight"])].copy()

    if len(group) == 0:
        continue

    x, y = weighted_gaussian_kde_1d(
        group["S_N2_chi7"].to_numpy(),
        group["comm_opes_weight"].to_numpy(),
        grid=np.linspace(0.0, 1.05, 400),
    )

    ax.plot(x, y, lw=2.0, label=f"{T} K parent reactive frame")

ax.axvline(THRESHOLD, ls=":", lw=1.0, label=r"$S=0.8$")
ax.set_xlabel(r"$S(N_2,\chi_7)$")
ax.set_ylabel("OPES-weighted probability density")
ax.set_title("Committor TS: isolated snapshots vs parent reactive frames")
ax.legend(frameon=False, fontsize=8)
plt.show()


# -------------------------------------------------------------------------
# Optional stricter p_c ≈ 0.5 comparison
# -------------------------------------------------------------------------

strict_parent = committor_parent_ts_sites[
    (committor_parent_ts_sites["p_diss"] >= COMM_STRICT_TS_MIN) &
    (committor_parent_ts_sites["p_diss"] <= COMM_STRICT_TS_MAX)
].copy()

print(
    f"Strict committor window: {COMM_STRICT_TS_MIN:.2f} <= p_diss <= {COMM_STRICT_TS_MAX:.2f}"
)

display(strict_parent.groupby("T").size().rename("n_strict_parent_TS"))

fig, ax = plt.subplots(figsize=(6.2, 3.8), dpi=140)

plot_weighted_similarity_kde_by_temperature(
    strict_parent,
    value_col="S_N2_chi7",
    weight_col="comm_opes_weight",
    ax=ax,
    title=(
        rf"Parent reactive frames: OPES-reweighted strict committor TS "
        rf"({COMM_STRICT_TS_MIN:.2f} $\leq p_\mathrm{{diss}} \leq$ {COMM_STRICT_TS_MAX:.2f})"
    ),
    label_suffix=" strict parent",
)

ax.set_xlabel(r"$S(N_2,\chi_7)$")
plt.show()


## 15. SOAP-based atomic-environment discovery

So far, the tutorial has used a **reference-based** active-site descriptor:

\[
S(\chi,\chi_7)
\]

This asks: *how similar is the local Fe environment to the known \(\chi_7\) active-site motif?*

In this optional section we add a more **agnostic, data-driven** analysis inspired by Cioni *et al.* on dynamic metal surfaces. The idea is to describe each local Fe environment with a SOAP fingerprint, reduce the dimensionality with PCA, and then cluster the environments without giving the algorithm the \(\chi_7\) label.

The scientific question is:

> Can SOAP discover an atomic environment that is enriched in \(\chi_7\)-like sites and in TS-like N$_2$ dissociation configurations?

The analysis has two parts:

```text
non-reactive Fe(111) trajectories
  -> SOAP of OVITO-selected surface Fe atoms
  -> PCA map and unsupervised clustering
  -> compare clusters with PLUMED S(chi, chi7)

reactive N2 trajectories
  -> SOAP of the local Fe atom below/around N2
  -> project into the same PCA/cluster space
  -> check whether TS-like configurations are enriched in specific SOAP clusters
```

In the Cioni workflow, the SOAP settings used for Cu surfaces were:

```python
SOAPrcut = 6
SOAPnmax = 8
SOAPlmax = 8
```

We use the same \(r_\mathrm{cut}\), \(n_\mathrm{max}\), and \(l_\mathrm{max}\) here, adapting the species from Cu to Fe.

### 15.1 Dependencies and SOAP parameters

The general `compcatschool` installation includes DScribe and scikit-learn for this section. HDBSCAN is optional; if it is unavailable, the notebook falls back to `KMeans`, which is simpler and robust for a teaching environment.

In [ ]:
# -------------------------------------------------------------------------
# 16.1 Dependencies and SOAP parameters
# -------------------------------------------------------------------------

try:
    from dscribe.descriptors import SOAP
except ImportError as exc:
    raise ImportError(
        "The SOAP section requires DScribe. "
        "Install the active-sites dependencies listed in the repository README."
    ) from exc

from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans

try:
    import hdbscan
    HDBSCAN_AVAILABLE = True
except ImportError:
    HDBSCAN_AVAILABLE = False

# SOAP parameters inspired by the Cioni et al. workflow.
SOAP_R_CUT = 6.0
SOAP_N_MAX = 8
SOAP_L_MAX = 8

# dscribe requires a Gaussian smearing width. SOAPify in the Cioni workflow
# hides some dscribe details; here we expose sigma explicitly.
SOAP_SIGMA = 0.5

# We use Fe-only SOAP for the main comparison. This tests whether the local
# surface geometry alone is sufficient to discover chi7-like environments.
SOAP_SPECIES = ["Fe"]

# To keep the tutorial fast, fit PCA/clustering on a random subset.
MAX_SOAP_SAMPLES_PER_TEMPERATURE = 4000
SOAP_RANDOM_SEED = 7

# Clustering options.
USE_HDBSCAN_IF_AVAILABLE = True
HDBSCAN_MIN_CLUSTER_SIZE = 200
KMEANS_N_CLUSTERS = 6

print("HDBSCAN available:", HDBSCAN_AVAILABLE)
print("SOAP species:", SOAP_SPECIES)
print("SOAP parameters:", SOAP_R_CUT, SOAP_N_MAX, SOAP_L_MAX, SOAP_SIGMA)

### 15.2 Helper functions for SOAP computation

The helper below computes SOAP descriptors for selected atoms in selected frames.

For the non-reactive analysis, the selected atoms are the recomputed OVITO upper-surface Fe atoms.

For the reactive analysis, the selected atom is the local Fe atom below/around N$_2$ that was already identified in point 15.

Important implementation detail: in the main analysis we use **Fe-only SOAP**:

```python
SOAP_SPECIES = ["Fe"]
```

The reactive trajectories, however, also contain N atoms. Recent `dscribe` versions require all atoms in the ASE `Atoms` object to belong to the descriptor species list, even if the SOAP centers are only Fe atoms. The helper therefore makes a temporary Fe-only copy of reactive frames and remaps the selected Fe center index before computing SOAP. This keeps the descriptor dimension consistent between non-reactive and reactive analyses.

In [ ]:
def compute_soap_for_table(traj_by_temperature, table, species=SOAP_SPECIES):
    """Compute normalized SOAP descriptors for rows containing T, frame, atom_id."""
    try:
        soap = SOAP(
            species=species,
            periodic=True,
            r_cut=SOAP_R_CUT,
            n_max=SOAP_N_MAX,
            l_max=SOAP_L_MAX,
            sigma=SOAP_SIGMA,
            sparse=False,
        )
    except TypeError:
        soap = SOAP(
            species=species,
            periodic=True,
            rcut=SOAP_R_CUT,
            nmax=SOAP_N_MAX,
            lmax=SOAP_L_MAX,
            sigma=SOAP_SIGMA,
            sparse=False,
        )

    descriptors = []
    rows = []
    species_set = set(species)

    for (T, frame), group in table.groupby(["T", "frame"], sort=True):
        atoms = traj_by_temperature[int(T)][int(frame)]
        centers = group["atom_id"].astype(int).to_list()
        symbols = np.array(atoms.get_chemical_symbols())

        if set(symbols).issubset(species_set):
            atoms_for_soap = atoms
            centers_for_soap = centers
        else:
            keep = np.array([symbol in species_set for symbol in symbols])
            kept_indices = np.where(keep)[0]
            old_to_new = {int(old): int(new) for new, old in enumerate(kept_indices)}

            missing = [center for center in centers if center not in old_to_new]
            if missing:
                bad = missing[0]
                raise ValueError(
                    f"SOAP center atom {bad} has symbol {symbols[bad]}, "
                    f"which is not included in SOAP_SPECIES={species}."
                )

            atoms_for_soap = atoms[kept_indices]
            centers_for_soap = [old_to_new[center] for center in centers]

        X_frame = np.asarray(soap.create(atoms_for_soap, centers=centers_for_soap))
        descriptors.append(X_frame)
        rows.append(group.copy())

    X = normalize(np.vstack(descriptors), norm="l2")
    out_table = pd.concat(rows, ignore_index=True)
    return X, out_table


def sample_surface_environments_for_soap(surface_table):
    """Sample surface environments for the SOAP/PCA/clustering fit."""
    sampled = []

    for T, group in surface_table.groupby("T"):
        n = min(len(group), MAX_SOAP_SAMPLES_PER_TEMPERATURE)
        sampled.append(
            group.sample(
                n=n,
                random_state=SOAP_RANDOM_SEED + int(T),
            )
        )

    return pd.concat(sampled, ignore_index=True)


traj_by_T = {
    300: traj300,
    700: traj700,
}



### 15.3 SOAP map of non-reactive Fe surface environments

We now compute SOAP descriptors for a representative subset of the recomputed OVITO surface atoms.

The output is:

```text
soap_surface_X      normalized SOAP vectors
soap_surface_table  metadata aligned with each SOAP vector
```

We then fit a 3D PCA representation, following the spirit of the Cioni bottom-up analysis.

In [ ]:
soap_surface_table = sample_surface_environments_for_soap(plumed_surface_values)

print("SOAP surface sample size:")
display(soap_surface_table.groupby("T").size())

soap_surface_X, soap_surface_table = compute_soap_for_table(
    traj_by_T,
    soap_surface_table,
    species=SOAP_SPECIES,
)

print("SOAP matrix shape:", soap_surface_X.shape)

soap_pca = PCA(n_components=3)
soap_surface_Z = soap_pca.fit_transform(soap_surface_X)

soap_surface_table = soap_surface_table.copy()
soap_surface_table["PC1"] = soap_surface_Z[:, 0]
soap_surface_table["PC2"] = soap_surface_Z[:, 1]
soap_surface_table["PC3"] = soap_surface_Z[:, 2]

print("Explained variance ratio:", soap_pca.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(6.2, 4.6), dpi=140)

for T, group in soap_surface_table.groupby("T"):
    ax.scatter(
        group["PC1"],
        group["PC2"],
        s=5,
        alpha=0.35,
        label=f"{T} K",
    )

ax.set_xlabel("SOAP PC1")
ax.set_ylabel("SOAP PC2")
ax.set_title("SOAP-PCA map of surface Fe environments")
ax.legend(frameon=False)
plt.show()

fig, ax = plt.subplots(figsize=(6.2, 4.6), dpi=140)

sc = ax.scatter(
    soap_surface_table["PC1"],
    soap_surface_table["PC2"],
    c=soap_surface_table["S_plumed"],
    s=5,
    alpha=0.45,
)
ax.set_xlabel("SOAP PC1")
ax.set_ylabel("SOAP PC2")
ax.set_title(r"SOAP map colored by PLUMED $S(\chi,\chi_7)$")
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label(r"$S(\chi,\chi_7)$")
plt.show()

### 15.4 Unsupervised clustering of SOAP environments

This step asks whether surface environments naturally organize into groups.

The Cioni bottom-up workflow uses HDBSCAN on the first PCA components. We do the same if `hdbscan` is installed. Otherwise, the notebook falls back to `KMeans`.
Plotting note: HDBSCAN marks noise/unassigned points as cluster `-1`. In the plot below, noise is shown in light gray and the real clusters are shown with a fixed categorical palette. This is easier to interpret than a continuous colorbar for integer cluster IDs.


In [ ]:
# -------------------------------------------------------------------------
# 15.4 Cluster the SOAP environments and plot them with categorical colors
# -------------------------------------------------------------------------

Z_for_clustering = soap_surface_table[["PC1", "PC2", "PC3"]].to_numpy()

if USE_HDBSCAN_IF_AVAILABLE and HDBSCAN_AVAILABLE:
    cluster_method = "HDBSCAN"
    soap_clusterer = hdbscan.HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples=None,
        cluster_selection_method="eom",
        prediction_data=True,
    )
    soap_labels = soap_clusterer.fit_predict(Z_for_clustering)
else:
    cluster_method = "KMeans"
    soap_clusterer = KMeans(
        n_clusters=KMEANS_N_CLUSTERS,
        random_state=SOAP_RANDOM_SEED,
        n_init="auto",
    )
    soap_labels = soap_clusterer.fit_predict(Z_for_clustering)

soap_surface_table["soap_cluster"] = soap_labels

print("Clustering method:", cluster_method)
print("Clusters found:", sorted(pd.unique(soap_labels)))

cluster_palette = [
    "#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9",
    "#F0E442", "#332288", "#117733", "#882255", "#44AA99", "#AA4499",
]

SOAP_CLUSTER_STYLE = {}
color_index = 0

for label in sorted(pd.unique(soap_surface_table["soap_cluster"])):
    if int(label) == -1:
        SOAP_CLUSTER_STYLE[label] = {
            "color": "#BDBDBD",
            "label": "noise / unassigned (-1)",
            "alpha": 0.18,
            "size": 5,
            "zorder": 1,
        }
    else:
        SOAP_CLUSTER_STYLE[label] = {
            "color": cluster_palette[color_index % len(cluster_palette)],
            "label": f"cluster {label}",
            "alpha": 0.70,
            "size": 8,
            "zorder": 2,
        }
        color_index += 1


def plot_soap_clusters(
    table,
    ax=None,
    pcx="PC1",
    pcy="PC2",
    cluster_col="soap_cluster",
    title=None,
    legend=True,
):
    """Scatter plot of SOAP clusters with categorical colors and a clean legend."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6.2, 4.6), dpi=140)

    for cluster, group in table.groupby(cluster_col):
        style = SOAP_CLUSTER_STYLE.get(
            cluster,
            {
                "color": "#444444",
                "label": f"cluster {cluster}",
                "alpha": 0.55,
                "size": 7,
                "zorder": 2,
            },
        )

        ax.scatter(
            group[pcx],
            group[pcy],
            s=style["size"],
            alpha=style["alpha"],
            color=style["color"],
            label=style["label"],
            linewidths=0,
            zorder=style["zorder"],
        )

    ax.set_xlabel(pcx.replace("PC", "SOAP PC"))
    ax.set_ylabel(pcy.replace("PC", "SOAP PC"))

    if title is not None:
        ax.set_title(title)

    if legend:
        ax.legend(
            frameon=False,
            markerscale=2.0,
            fontsize=8,
            ncol=2 if table[cluster_col].nunique() > 6 else 1,
            loc="best",
        )

    return ax


fig, ax = plt.subplots(figsize=(6.4, 4.8), dpi=140)

plot_soap_clusters(
    soap_surface_table,
    ax=ax,
    title=f"Unsupervised SOAP clusters ({cluster_method})",
)

plt.show()



### 15.5 Are any SOAP clusters enriched in \(\chi_7\)-like sites?

This is the key validation step.

SOAP clustering is agnostic: it does not know the \(\chi_7\) reference. We therefore compare each SOAP cluster against the PLUMED similarity:

```text
mean S(chi, chi7)
fraction with S >= 0.8
temperature composition
```

If one cluster has a high fraction of \(\chi_7\)-like atoms, SOAP has discovered an environment close to the active-site motif.

In [ ]:
cluster_summary = (
    soap_surface_table
    .groupby("soap_cluster")
    .agg(
        n=("atom_id", "count"),
        mean_S_chi7=("S_plumed", "mean"),
        median_S_chi7=("S_plumed", "median"),
        fraction_chi7_like=("active_plumed", "mean"),
        fraction_700K=("T", lambda x: np.mean(x == 700)),
        mean_z=("z", "mean"),
    )
    .sort_values("mean_S_chi7", ascending=False)
)

display(cluster_summary)

chi7_enriched_clusters = (
    cluster_summary
    .sort_values("mean_S_chi7", ascending=False)
    .head(2)
    .index
    .to_list()
)

print("Most chi7-enriched SOAP clusters:", chi7_enriched_clusters)

fig, ax = plt.subplots(figsize=(6.4, 3.8), dpi=140)

cluster_summary.sort_index()["fraction_chi7_like"].plot(kind="bar", ax=ax)
ax.set_xlabel("SOAP cluster")
ax.set_ylabel(r"Fraction with $S(\chi,\chi_7)\geq0.8$")
ax.set_title(r"Which SOAP clusters are enriched in $\chi_7$-like sites?")
plt.show()

fig, ax = plt.subplots(figsize=(6.4, 3.8), dpi=140)

clusters_sorted = sorted(soap_surface_table["soap_cluster"].unique())
data_to_plot = [
    soap_surface_table.loc[soap_surface_table["soap_cluster"] == cluster, "S_plumed"].dropna().to_numpy()
    for cluster in clusters_sorted
]
labels = [str(c) for c in clusters_sorted]

ax.violinplot(data_to_plot, showmedians=True, showextrema=False)
ax.axhline(THRESHOLD, ls="--", lw=1.0)
ax.set_xticks(np.arange(1, len(labels) + 1))
ax.set_xticklabels(labels)
ax.set_xlabel("SOAP cluster")
ax.set_ylabel(r"$S(\chi,\chi_7)$")
ax.set_title(r"PLUMED similarity distribution within SOAP clusters")
plt.show()

### 15.6 Reactive trajectories: SOAP cluster of the local Fe atom near N$_2$

We now take the local Fe atom below/around N$_2$ from the reactive analysis in point 15 and compute its Fe-only SOAP descriptor.

Then we project these reactive local environments into the same PCA space fitted on the non-reactive surface environments.

This asks:

> Do TS-like N$_2$ configurations fall into the same SOAP clusters that are enriched in \(\chi_7\)-like surface motifs?
Important weighting note: SOAP vectors, PCA projection and cluster assignment are deterministic structural operations and are not themselves reweighted. However, any **population**, **fraction**, **cluster enrichment**, or **distribution** computed from OPES-biased reactive configurations must use OPES weights.


In [ ]:
def assign_soap_clusters(Z):
    """Assign clusters to projected SOAP-PCA points."""
    if cluster_method == "HDBSCAN":
        labels, strengths = hdbscan.approximate_predict(soap_clusterer, Z)
        return labels
    return soap_clusterer.predict(Z)


reactive_soap_input = n2_sites[np.isfinite(n2_sites["site_atom_id"])].copy()
reactive_soap_input["atom_id"] = reactive_soap_input["site_atom_id"].astype(int)

reactive_traj_by_T = {
    300: reactive_trajs[300],
    700: reactive_trajs[700],
}

# With SOAP_SPECIES = ["Fe"], compute_soap_for_table temporarily removes N atoms
# from each reactive frame and remaps the selected Fe center index.
reactive_soap_X, reactive_soap_table = compute_soap_for_table(
    reactive_traj_by_T,
    reactive_soap_input,
    species=SOAP_SPECIES,
)

reactive_soap_Z = soap_pca.transform(reactive_soap_X)

reactive_soap_table = reactive_soap_table.copy()
reactive_soap_table["PC1"] = reactive_soap_Z[:, 0]
reactive_soap_table["PC2"] = reactive_soap_Z[:, 1]
reactive_soap_table["PC3"] = reactive_soap_Z[:, 2]
reactive_soap_table["soap_cluster"] = assign_soap_clusters(reactive_soap_Z)

display(reactive_soap_table.head())

# -------------------------------------------------------------------------
# Unweighted cluster summary: useful as a raw diagnostic only.
# -------------------------------------------------------------------------

reactive_cluster_summary = (
    reactive_soap_table
    .groupby(["T", "soap_cluster"])
    .agg(
        n=("frame", "count"),
        fraction_ts_like=("ts_like", "mean"),
        fraction_high_charge_ts_like=("high_charge_ts_like", "mean"),
        mean_abs_q_N2=("abs_q_N2", "mean"),
        mean_d_NN_A=("d_NN_A", "mean"),
        mean_S_N2_chi7=("S_N2_chi7", "mean"),
        fraction_chi7_like=("active_site_below_N2", "mean"),
    )
    .reset_index()
    .sort_values(["T", "fraction_ts_like", "n"], ascending=[True, False, False])
)

print("Raw/unweighted reactive SOAP cluster summary:")
display(reactive_cluster_summary)


# -------------------------------------------------------------------------
# OPES-weighted cluster summary: use this for statistical interpretation.
# -------------------------------------------------------------------------

if "opes_weight" in reactive_soap_table.columns:
    weighted_reactive_cluster_rows = []

    for (T, cluster), group in reactive_soap_table.groupby(["T", "soap_cluster"]):
        weights = group["opes_weight"].to_numpy()
        finite_weights = weights[np.isfinite(weights) & (weights > 0)]

        weighted_reactive_cluster_rows.append(
            {
                "T": int(T),
                "soap_cluster": int(cluster),
                "n_raw": len(group),
                "N_eff": effective_sample_size(weights),
                "weighted_population": float(finite_weights.sum()) if len(finite_weights) else np.nan,
                "weighted_fraction_ts_like": weighted_mean(group["ts_like"].astype(float), weights),
                "weighted_fraction_high_charge_ts_like": weighted_mean(
                    group["high_charge_ts_like"].astype(float),
                    weights,
                ),
                "weighted_mean_abs_q_N2": weighted_mean(group["abs_q_N2"], weights),
                "weighted_mean_d_NN_A": weighted_mean(group["d_NN_A"], weights),
                "weighted_mean_S_N2_chi7": weighted_mean(group["S_N2_chi7"], weights),
                "weighted_fraction_chi7_like": weighted_mean(
                    group["active_site_below_N2"].astype(float),
                    weights,
                ),
            }
        )

    reactive_cluster_summary_weighted = pd.DataFrame(weighted_reactive_cluster_rows)

    if len(reactive_cluster_summary_weighted) > 0:
        reactive_cluster_summary_weighted["weighted_population_fraction_within_T"] = (
            reactive_cluster_summary_weighted
            .groupby("T")["weighted_population"]
            .transform(lambda x: x / x.sum() if x.sum() > 0 else np.nan)
        )

    print("OPES-weighted reactive SOAP cluster summary:")
    display(
        reactive_cluster_summary_weighted
        .sort_values(["T", "weighted_population_fraction_within_T"], ascending=[True, False])
    )

else:
    print(
        "No opes_weight column found. Run section 14.7b before this SOAP analysis "
        "to obtain OPES-weighted cluster statistics."
    )



### 15.7 Does SOAP recover the active environment sampled by N$_2$?

The most useful comparison is between:

1. SOAP clusters enriched in \(\chi_7\)-like sites in the non-reactive surface trajectories;
2. SOAP clusters sampled by TS-like reactive configurations.

A positive signal would be:

```text
TS-like reactive configurations preferentially fall into
one or more clusters with high mean S(chi, chi7)
```

This would suggest that SOAP can recover the active environment in a more agnostic way.
In the plots and tables below, the raw counts are kept as diagnostics. The OPES-weighted cluster populations are the appropriate quantities for comparing how frequently different SOAP environments are sampled in the biased reactive trajectories.


In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.8), dpi=140)

# Draw the non-reactive surface SOAP clusters as a lightly transparent background.
plot_soap_clusters(
    soap_surface_table,
    ax=ax,
    title="TS-like N2 environments projected on the SOAP surface map",
    legend=False,
)

# Make the background lighter without changing the cluster identity.
for collection in ax.collections:
    collection.set_alpha(min(collection.get_alpha() or 1.0, 0.18))
    collection.set_sizes([4])

ts_reactive = reactive_soap_table[reactive_soap_table["ts_like"]].copy()

for T, group in ts_reactive.groupby("T"):
    ax.scatter(
        group["PC1"],
        group["PC2"],
        s=32,
        alpha=0.9,
        marker="x",
        linewidths=1.4,
        color="black" if int(T) == 300 else "#D55E00",
        label=f"{T} K TS-like N2",
        zorder=5,
    )

ax.set_xlabel("SOAP PC1")
ax.set_ylabel("SOAP PC2")
ax.legend(frameon=False, fontsize=8)
plt.show()


# -------------------------------------------------------------------------
# Raw counts of TS-like frames per SOAP cluster.
# -------------------------------------------------------------------------

ts_cluster_counts = (
    ts_reactive
    .groupby(["T", "soap_cluster"])
    .size()
    .rename("n_ts_like_raw")
    .reset_index()
)

display(ts_cluster_counts)


# -------------------------------------------------------------------------
# OPES-weighted TS-like populations per SOAP cluster.
# -------------------------------------------------------------------------

if "opes_weight" in ts_reactive.columns:
    ts_cluster_weighted_rows = []

    for (T, cluster), group in ts_reactive.groupby(["T", "soap_cluster"]):
        weights = group["opes_weight"].to_numpy()
        finite_weights = weights[np.isfinite(weights) & (weights > 0)]

        ts_cluster_weighted_rows.append(
            {
                "T": int(T),
                "soap_cluster": int(cluster),
                "n_ts_like_raw": len(group),
                "N_eff": effective_sample_size(weights),
                "weighted_TS_population": float(finite_weights.sum()) if len(finite_weights) else np.nan,
                "weighted_mean_abs_q_N2": weighted_mean(group["abs_q_N2"], weights),
                "weighted_mean_d_NN_A": weighted_mean(group["d_NN_A"], weights),
                "weighted_mean_S_N2_chi7": weighted_mean(group["S_N2_chi7"], weights),
            }
        )

    ts_cluster_counts_weighted = pd.DataFrame(ts_cluster_weighted_rows)

    if len(ts_cluster_counts_weighted) > 0:
        ts_cluster_counts_weighted["weighted_TS_fraction_within_T"] = (
            ts_cluster_counts_weighted
            .groupby("T")["weighted_TS_population"]
            .transform(lambda x: x / x.sum() if x.sum() > 0 else np.nan)
        )

    print("OPES-weighted TS-like population by SOAP cluster:")
    display(
        ts_cluster_counts_weighted
        .sort_values(["T", "weighted_TS_fraction_within_T"], ascending=[True, False])
    )

    fig, ax = plt.subplots(figsize=(6.4, 3.8), dpi=140)

    for T, group in ts_cluster_counts_weighted.groupby("T"):
        group = group.sort_values("soap_cluster")
        ax.plot(
            group["soap_cluster"].astype(str),
            group["weighted_TS_fraction_within_T"],
            marker="o",
            lw=1.5,
            label=f"{T} K",
        )

    ax.set_xlabel("SOAP cluster")
    ax.set_ylabel("OPES-weighted TS fraction within T")
    ax.set_title("Which SOAP clusters are sampled in the TS-like region?")
    ax.legend(frameon=False)
    plt.show()

else:
    print("No opes_weight column found; plotting raw TS-like counts.")

    fig, ax = plt.subplots(figsize=(6.4, 3.8), dpi=140)

    for T, group in ts_cluster_counts.groupby("T"):
        ax.plot(
            group["soap_cluster"].astype(str),
            group["n_ts_like_raw"],
            marker="o",
            lw=1.5,
            label=f"{T} K",
        )

    ax.set_xlabel("SOAP cluster")
    ax.set_ylabel("Number of TS-like frames")
    ax.set_title("Which SOAP clusters are sampled in the TS-like region?")
    ax.legend(frameon=False)
    plt.show()


cluster_active_score = (
    cluster_summary[["mean_S_chi7", "fraction_chi7_like"]]
    .rename_axis("soap_cluster")
    .reset_index()
)

if "ts_cluster_counts_weighted" in globals() and len(ts_cluster_counts_weighted) > 0:
    ts_sampling_score = (
        ts_cluster_counts_weighted
        .groupby("soap_cluster")
        .agg(
            n_ts_like_raw=("n_ts_like_raw", "sum"),
            weighted_TS_population=("weighted_TS_population", "sum"),
            mean_weighted_TS_fraction_within_T=("weighted_TS_fraction_within_T", "mean"),
            weighted_mean_abs_q_N2=("weighted_mean_abs_q_N2", "mean"),
            weighted_mean_d_NN_A=("weighted_mean_d_NN_A", "mean"),
            weighted_mean_S_N2_chi7=("weighted_mean_S_N2_chi7", "mean"),
        )
        .reset_index()
    )
else:
    ts_sampling_score = (
        ts_reactive
        .groupby("soap_cluster")
        .agg(
            n_ts_like_raw=("frame", "count"),
            mean_abs_q_N2=("abs_q_N2", "mean"),
            mean_d_NN_A=("d_NN_A", "mean"),
            mean_S_N2_chi7=("S_N2_chi7", "mean"),
        )
        .reset_index()
    )

soap_reactivity_comparison = cluster_active_score.merge(
    ts_sampling_score,
    on="soap_cluster",
    how="left",
).fillna({"n_ts_like_raw": 0, "weighted_TS_population": 0})

display(
    soap_reactivity_comparison
    .sort_values(
        ["weighted_TS_population", "n_ts_like_raw", "mean_S_chi7"],
        ascending=False,
    )
)

print(
    "Interpretation guide: clusters with high fraction_chi7_like and high "
    "OPES-weighted TS population are candidate SOAP-discovered active environments."
)

### 15.7b SOAP clusters for the committor-selected TS ensemble

The previous SOAP analysis projected the local Fe environment near N$_2$ from the reactive trajectory subset selected by a geometric TS-like window.

Here we do the stricter comparison: we take the **committor-selected TS snapshots**, compute Fe-only SOAP for the local Fe atom below/around N$_2$, project these environments into the same SOAP-PCA space, and assign the same SOAP clusters.

This asks the same question as before, but on the transition-state ensemble selected by committor analysis:

> Are committor-TS configurations enriched in the same SOAP clusters that are also enriched in \(\chi_7\)-like environments?
The same weighting logic applies here: the SOAP projection and cluster assignment are structural descriptors, but the cluster populations of the committor-selected configurations should be OPES-weighted because the initial snapshots were selected from a biased parent trajectory.


In [ ]:
# -------------------------------------------------------------------------
# 15.7b SOAP clusters for committor-selected TS snapshots
# -------------------------------------------------------------------------

if "committor_ts_sites" not in globals():
    raise RuntimeError("Run section 14.9 before this SOAP committor analysis.")

committor_soap_input = committor_ts_sites[np.isfinite(committor_ts_sites["site_atom_id"])].copy()
committor_soap_input["atom_id"] = committor_soap_input["site_atom_id"].astype(int)

# compute_soap_for_table works with SOAP_SPECIES=["Fe"] because it temporarily
# removes N atoms and remaps the selected Fe center index.
committor_soap_X, committor_soap_table = compute_soap_for_table(
    committor_ts_trajs,
    committor_soap_input,
    species=SOAP_SPECIES,
)

committor_soap_Z = soap_pca.transform(committor_soap_X)
committor_soap_table = committor_soap_table.copy()
committor_soap_table["PC1"] = committor_soap_Z[:, 0]
committor_soap_table["PC2"] = committor_soap_Z[:, 1]
committor_soap_table["PC3"] = committor_soap_Z[:, 2]
committor_soap_table["soap_cluster"] = assign_soap_clusters(committor_soap_Z)

committor_soap_summary = (
    committor_soap_table
    .groupby(["T", "soap_cluster"])
    .agg(
        n_TS=("frame", "count"),
        mean_p_diss=("p_diss", "mean"),
        mean_d_NN_A=("d_NN_A", "mean"),
        mean_S_N2_chi7=("S_N2_chi7", "mean"),
        fraction_chi7_like=("active_site_below_N2", "mean"),
    )
    .reset_index()
    .sort_values(["T", "n_TS"], ascending=[True, False])
)

display(committor_soap_summary)

fig, ax = plt.subplots(figsize=(6.4, 4.8), dpi=140)

plot_soap_clusters(
    soap_surface_table,
    ax=ax,
    title="Committor TS environments projected on the SOAP surface map",
    legend=False,
)

for collection in ax.collections:
    collection.set_alpha(min(collection.get_alpha() or 1.0, 0.16))
    collection.set_sizes([4])

for T, group in committor_soap_table.groupby("T"):
    ax.scatter(
        group["PC1"],
        group["PC2"],
        s=30,
        alpha=0.85,
        marker="x",
        linewidths=1.3,
        color="black" if int(T) == 300 else "#D55E00",
        label=f"{T} K committor TS",
        zorder=5,
    )

ax.set_xlabel("SOAP PC1")
ax.set_ylabel("SOAP PC2")
ax.legend(frameon=False, fontsize=8)
plt.show()

# Compare chi7 enrichment and committor TS sampling in one table.
cluster_active_score = (
    cluster_summary[["mean_S_chi7", "fraction_chi7_like"]]
    .rename_axis("soap_cluster")
    .reset_index()
)

comm_ts_sampling_score = (
    committor_soap_table
    .groupby("soap_cluster")
    .agg(
        n_committor_TS=("frame", "count"),
        mean_p_diss=("p_diss", "mean"),
        mean_d_NN_A=("d_NN_A", "mean"),
        mean_S_N2_chi7=("S_N2_chi7", "mean"),
    )
    .reset_index()
)

committor_soap_reactivity_comparison = cluster_active_score.merge(
    comm_ts_sampling_score,
    on="soap_cluster",
    how="left",
).fillna({"n_committor_TS": 0})

display(
    committor_soap_reactivity_comparison
    .sort_values(["n_committor_TS", "mean_S_chi7"], ascending=False)
)


# -------------------------------------------------------------------------
# Optional OPES-weighted cluster enrichment
# -------------------------------------------------------------------------

if "comm_opes_weight" in committor_soap_table.columns:
    weighted_cluster_rows = []

    for (T, cluster), group in committor_soap_table.groupby(["T", "soap_cluster"]):
        weights = group["comm_opes_weight"].to_numpy()
        finite_weights = weights[np.isfinite(weights) & (weights > 0)]

        weighted_cluster_rows.append(
            {
                "T": int(T),
                "soap_cluster": int(cluster),
                "n_TS": len(group),
                "N_eff": effective_sample_size(weights),
                "weighted_TS_population": float(finite_weights.sum()) if len(finite_weights) else np.nan,
                "weighted_mean_p_diss": weighted_mean(group["p_diss"], weights),
                "weighted_mean_d_NN_A": weighted_mean(group["d_NN_A"], weights),
                "weighted_mean_S_N2_chi7": weighted_mean(group["S_N2_chi7"], weights),
                "weighted_fraction_chi7_like": weighted_mean(
                    group["active_site_below_N2"].astype(float),
                    weights,
                ),
            }
        )

    committor_soap_weighted_summary = pd.DataFrame(weighted_cluster_rows)

    # Normalize weighted population within each temperature.
    if len(committor_soap_weighted_summary) > 0:
        committor_soap_weighted_summary["weighted_TS_fraction_within_T"] = (
            committor_soap_weighted_summary
            .groupby("T")["weighted_TS_population"]
            .transform(lambda x: x / x.sum() if x.sum() > 0 else np.nan)
        )

    print("OPES-weighted SOAP-cluster summary for committor TS snapshots:")
    display(
        committor_soap_weighted_summary
        .sort_values(["T", "weighted_TS_fraction_within_T"], ascending=[True, False])
    )
else:
    print(
        "No comm_opes_weight column found. Run section 14.9b before this cell "
        "to obtain OPES-weighted SOAP-cluster statistics."
    )



### 15.8 Export representative reactive TS-like frames for selected SOAP clusters

This cell extracts representative reactive frames for selected SOAP clusters.

The default request is:

```text
T = 300 K, SOAP cluster = 4
T = 300 K, SOAP cluster = -1
T = 700 K, SOAP cluster = 4
T = 700 K, SOAP cluster = -1
```

For each case, the notebook tries to select a **TS-like** configuration first:

\[
1.6 \le d_{\mathrm{NN}} \le 1.8\ \mathrm{\AA}.
\]

If no TS-like frame exists for a given temperature/cluster pair, the code falls back to all frames in that cluster and prints a warning.

Each exported frame contains the full reactive structure and extra per-atom arrays that can be used in OVITO or ASE:

```text
highlight = 0  other atoms
highlight = 1  Fe atoms within 3 Å of N2
highlight = 2  local Fe atom below/around N2
highlight = 3  N atoms of N2

is_fe_around_N2
is_local_Fe_site
is_N2_atom
site_soap_cluster
```

The cell also exports a small local cutout around N2 and the reference \(\chi_7\) environment, which is useful for visual comparison with the \(\chi_7\)-type and defected structures discussed in Bonati *et al.* Fig. 7.

In [ ]:
# -------------------------------------------------------------------------
# User settings
# -------------------------------------------------------------------------

TARGET_SOAP_CLUSTERS = [4, -1]
TARGET_TEMPERATURES = [300, 700]

# Use TS-like frames first; if no frame exists for a given pair, fallback to all frames.
EXPORT_TS_LIKE_FIRST = True

# Fe atoms within this distance from either N atom are highlighted.
LOCAL_N2_CUTOFF_A = 3.0

# Choose one representative frame per (temperature, cluster).
N_EXAMPLES_PER_CASE = 1

EXPORT_SOAP_FRAME_DIR = ROOT / "soap_cluster_reactive_frames"
EXPORT_SOAP_FRAME_DIR.mkdir(exist_ok=True)


def select_representative_reactive_rows(
    table,
    temperature,
    soap_cluster,
    ts_like_first=True,
    n_examples=1,
):
    """Select representative frames for one temperature and one SOAP cluster.

    The preferred selection is the TS-like subset. Within the selected subset,
    frames are sorted by closeness to the center of the TS-like dNN window.
    """
    subset = table[
        (table["T"] == temperature) &
        (table["soap_cluster"] == soap_cluster)
    ].copy()

    if len(subset) == 0:
        print(f"No frames found for T={temperature} K, SOAP cluster={soap_cluster}.")
        return subset

    used_ts_filter = False

    if ts_like_first and "ts_like" in subset.columns:
        ts_subset = subset[subset["ts_like"]].copy()
        if len(ts_subset) > 0:
            subset = ts_subset
            used_ts_filter = True
        else:
            print(
                f"Warning: no TS-like frames for T={temperature} K, "
                f"SOAP cluster={soap_cluster}. Falling back to all frames in this cluster."
            )

    ts_center = 0.5 * (TS_MIN_DNN + TS_MAX_DNN)

    subset["distance_to_TS_center"] = np.abs(subset["d_NN_A"] - ts_center)

    # Prefer configurations close to dNN = 1.7 Å.
    # If there is a tie, prefer high charge and then high similarity.
    sort_cols = ["distance_to_TS_center"]
    ascending = [True]

    if "abs_q_N2" in subset.columns:
        sort_cols.append("abs_q_N2")
        ascending.append(False)

    if "S_N2_chi7" in subset.columns:
        sort_cols.append("S_N2_chi7")
        ascending.append(False)

    subset = subset.sort_values(sort_cols, ascending=ascending).head(n_examples)

    print(
        f"Selected {len(subset)} frame(s) for T={temperature} K, "
        f"SOAP cluster={soap_cluster}, TS-like filter used={used_ts_filter}."
    )

    return subset


def annotate_reactive_frame_for_export(atoms, row, local_cutoff_A=3.0):
    """Copy and annotate a reactive frame for visualization.

    The exported frame preserves the full structure but adds arrays that can be
    used in OVITO or other viewers to color/highlight atoms.
    """
    atoms_out = atoms.copy()

    symbols = np.array(atoms_out.get_chemical_symbols())
    n_ids = np.where(symbols == "N")[0]
    fe_ids = np.where(symbols == "Fe")[0]

    if len(n_ids) != 2:
        raise ValueError(f"Expected two N atoms, found {len(n_ids)}.")

    distances = atoms_out.get_all_distances(mic=True)
    around_n2 = (distances[n_ids[:, None], fe_ids] < local_cutoff_A).any(axis=0)
    fe_around_n2 = fe_ids[around_n2]

    site_atom_id = int(row["site_atom_id"])
    soap_cluster = int(row["soap_cluster"])

    highlight = np.zeros(len(atoms_out), dtype=np.int32)
    is_fe_around_n2 = np.zeros(len(atoms_out), dtype=np.int32)
    is_local_site = np.zeros(len(atoms_out), dtype=np.int32)
    is_n2_atom = np.zeros(len(atoms_out), dtype=np.int32)
    site_soap_cluster = np.full(len(atoms_out), -999, dtype=np.int32)

    highlight[fe_around_n2] = 1
    is_fe_around_n2[fe_around_n2] = 1

    highlight[site_atom_id] = 2
    is_local_site[site_atom_id] = 1
    site_soap_cluster[site_atom_id] = soap_cluster

    highlight[n_ids] = 3
    is_n2_atom[n_ids] = 1

    atoms_out.arrays["highlight"] = highlight
    atoms_out.arrays["is_fe_around_N2"] = is_fe_around_n2
    atoms_out.arrays["is_local_Fe_site"] = is_local_site
    atoms_out.arrays["is_N2_atom"] = is_n2_atom
    atoms_out.arrays["site_soap_cluster"] = site_soap_cluster

    atoms_out.info.update(
        {
            "T_K": int(row["T"]),
            "notebook_frame": int(row["frame"]),
            "source_frame": int(row["source_frame"]) if "source_frame" in row else int(row["frame"]),
            "soap_cluster": soap_cluster,
            "d_NN_A": float(row["d_NN_A"]),
            "abs_q_N2": float(row["abs_q_N2"]),
            "S_N2_chi7": float(row["S_N2_chi7"]),
            "site_atom_id": site_atom_id,
            "local_N2_cutoff_A": float(local_cutoff_A),
        }
    )

    return atoms_out, fe_around_n2, n_ids


def make_local_cutout(atoms, selected_indices, center_index=None):
    """Make a small local structure for easier visual inspection."""
    selected_indices = np.array(sorted(set(int(i) for i in selected_indices)), dtype=int)
    local = atoms[selected_indices].copy()

    if center_index is not None and center_index in selected_indices:
        local_center_id = int(np.where(selected_indices == center_index)[0][0])
        local.positions -= local.positions[local_center_id]

    local.info.update(atoms.info)
    return local


exported_rows = []
exported_atoms = {}

for T in TARGET_TEMPERATURES:
    for cluster in TARGET_SOAP_CLUSTERS:
        rows = select_representative_reactive_rows(
            reactive_soap_table,
            temperature=T,
            soap_cluster=cluster,
            ts_like_first=EXPORT_TS_LIKE_FIRST,
            n_examples=N_EXAMPLES_PER_CASE,
        )

        for _, row in rows.iterrows():
            frame = int(row["frame"])
            atoms = reactive_trajs[int(T)][frame]

            atoms_annotated, fe_around_n2, n_ids = annotate_reactive_frame_for_export(
                atoms,
                row,
                local_cutoff_A=LOCAL_N2_CUTOFF_A,
            )

            source_frame = int(row["source_frame"]) if "source_frame" in row else frame
            cluster_label = str(int(cluster)).replace("-", "m")

            base = f"reactive_TS_T{int(T)}K_SOAPcluster_{cluster_label}_frame{source_frame}"

            full_xyz = EXPORT_SOAP_FRAME_DIR / f"{base}.xyz"
            local_xyz = EXPORT_SOAP_FRAME_DIR / f"{base}_local_N2_environment.xyz"

            write(full_xyz, atoms_annotated, format="extxyz")

            local_indices = list(fe_around_n2) + list(n_ids)
            local_atoms = make_local_cutout(
                atoms_annotated,
                local_indices,
                center_index=int(row["site_atom_id"]),
            )
            write(local_xyz, local_atoms, format="extxyz")

            exported_rows.append(
                {
                    "T": int(T),
                    "soap_cluster": int(cluster),
                    "frame": frame,
                    "source_frame": source_frame,
                    "d_NN_A": float(row["d_NN_A"]),
                    "abs_q_N2": float(row["abs_q_N2"]),
                    "S_N2_chi7": float(row["S_N2_chi7"]),
                    "site_atom_id": int(row["site_atom_id"]),
                    "full_xyz": str(full_xyz),
                    "local_xyz": str(local_xyz),
                }
            )

            exported_atoms[(int(T), int(cluster))] = atoms_annotated

# Export the chi7 reference environment for visual comparison.
chi7_reference_xyz = EXPORT_SOAP_FRAME_DIR / "chi7_reference_environment.xyz"
write(chi7_reference_xyz, ref_atoms, format="extxyz")

exported_frame_table = pd.DataFrame(exported_rows)

print("Exported representative frames:")
display(exported_frame_table)

print("Files written in:", EXPORT_SOAP_FRAME_DIR)
print("Reference chi7 environment:", chi7_reference_xyz)



### 15.8b Why we do not compute reactive SOAP-cluster lifetimes

The reactive N\(_2\) trajectories were generated with OPES bias. Consecutive residence times of SOAP clusters along those trajectories are therefore **not physical lifetimes**.

We keep the reactive SOAP analysis for:

```text
structural assignment
OPES-weighted cluster populations
OPES-weighted TS enrichment
```

but we do not report reactive cluster lifetimes.

For real lifetime analysis, use unbiased/non-reactive trajectories or a dedicated kinetic analysis.

### 15.9 Suggested extensions

Possible extensions for advanced students:

1. repeat the SOAP clustering with different `min_cluster_size` values;
2. include N in the reactive SOAP descriptor, i.e. use `SOAP_SPECIES = ["Fe", "N"]`;
3. build transition matrices between SOAP clusters using unbiased trajectories;
4. compute SOAP-cluster lifetimes only from unbiased/non-reactive trajectories;
5. compare the committor-selected ensemble with a stricter window around \(p_\mathrm{diss}=0.5\).

## 16. Exercises

1. Change the active-site threshold from `0.80` to `0.75` or `0.85`. Which conclusions are robust?
2. Compare the raw and OPES-reweighted distributions of `S_N2_chi7`. Does reweighting change the 300 K vs 700 K comparison?
3. In the committor section, inspect frames with `0.25 <= p_diss <= 0.75`. Are these transition-state snapshots enriched in \(\chi_7\)-like environments?
4. In the SOAP section, identify which clusters are enriched in active sites. Do the unsupervised SOAP clusters recover the PLUMED-defined motif?
5. For one representative SOAP cluster, export a local frame and describe the local Fe/N geometry in chemical terms.
